# Notebook 09 · Tout le code du projet, et les concepts derrière

Ce notebook rassemble **tout le code Python du projet** (dossiers `src/`, `app/` et `notebooks/06_annexe_news_prix/`),
dans l'ordre où il sert. Avant chaque fichier : ce qu'il fait, les concepts qu'il utilise expliqués simplement, et la
séance du cours qui les introduit (S = Applied ML, L = Maths & Stats).

Le code est **affiché, pas exécuté** : chaque cellule relit le fichier du dépôt, si bien que ce notebook montre toujours
la version à jour. Pour lancer un module : `python -m src.<module>` depuis la racine du projet.

| Partie | Contenu |
|---|---|
| 0 | Le projet en une page, et le lexique de tous les concepts |
| 1 | Configuration et données (dont Bloomberg) |
| 2 | Modèles, réglage, calibration, décision, test |
| 3 | Explicabilité et prédiction |
| 4 | Seconde version Bloomberg et améliorations sans surapprentissage |
| 5 | Explorations après coup : bancs d'essai, volatilité, signal de risque |
| 6 | Annexe news × prix (HMM, backtests) |
| 7 | La plateforme : API, tableau de bord, site |

In [1]:
from pathlib import Path
import json
from IPython.display import Code, Markdown, display

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def montrer(chemin):
    '''Affiche un fichier du dépôt avec la coloration Python, précédé de sa taille.'''
    f = RACINE / chemin
    texte = f.read_text(encoding="utf-8")
    display(Markdown(f"**`{chemin}`** · {len(texte.splitlines())} lignes"))
    display(Code(texte, language="python"))

def lire_json(chemin):
    return json.loads((RACINE / chemin).read_text(encoding="utf-8"))

fichiers = sorted(p.relative_to(RACINE).as_posix() for d in ["src", "app", "notebooks/06_annexe_news_prix"]
                  for p in (RACINE / d).glob("*.py"))
print(len(fichiers), "fichiers Python,", sum(len((RACINE / f).read_text().splitlines()) for f in fichiers), "lignes")

29 fichiers Python, 5971 lignes


## 0 · Le projet en une page

**La question.** Chaque fin de mois, faut-il acheter le Nikkei 225, rester neutre ou le vendre pour le mois suivant, et
faut-il couvrir le risque de change yen/euro ?

**La chaîne.**
1. *Collecte* (notebooks 00a, 00b) : prix, taux, change, enquêtes (Tankan), news GDELT notées par FinBERT, puis
   Bloomberg comme référence pour la seconde version. Chaque série est décalée à sa **date de publication**.
2. *Table de modélisation* (`src/data.py`) : une ligne par mois, les cibles du mois suivant, un audit des fuites.
3. *Découpage chronologique* : entraînement jusqu'à fin 2021, validation 2022-2023, test 2024-2026 ouvert une seule fois.
4. *Trois modèles* (logistique, forêt aléatoire, gradient boosting) dans une même pipeline, réglés par Optuna avec une
   validation croisée temporelle (`src/pipeline.py`, `src/tune.py`).
5. *Calibration de Platt et règle de décision* par espérance de gain, figées avant le test (`src/decision.py`).
6. *Test unique* (`src/evaluate.py`), puis *explicabilité* : tranches, permutation, SHAP (`src/predict.py`, notebook 04).
7. *Après coup* : seconde version 1999-2026 avec Bloomberg (`src/v2_bloomberg.py`), améliorations sous protocole figé
   (`src/ameliorations.py`), et prévision de la volatilité plutôt que du sens (`src/volatilite.py`, `src/signal_risque.py`).

## 0 · Lexique des concepts

### Cadrer et préparer les données (S1)
- **Fuite d'information (leakage).** Utiliser, pour prédire le mois t, une donnée qui n'était pas connue à la fin du
  mois t. Exemple du projet : le Tankan est daté par Bloomberg à la fin du trimestre, mais publié le mois suivant ;
  pris à sa date Bloomberg, il aurait fui. D'où le décalage de chaque série à sa date de publication, et un **audit
  automatique** qui arrête la construction si une variable regarde le futur.
- **Découpage chronologique.** En série temporelle, on n'apprend que sur le passé et on évalue sur le futur ; un
  mélange aléatoire ferait apprendre l'avenir. Trois blocs : entraînement, validation (pour choisir), test (pour juger,
  une seule fois).
- **Cible.** `y_baisse` = 1 si le Nikkei baisse le mois suivant (la classe positive des métriques est la baisse) ;
  `y_non_couvert` = 1 si ne pas couvrir le yen était meilleur.
- **Variables construites.** Momentum (rendement passé sur 3 et 12 mois), volatilité EWMA (moyenne exponentielle des
  rendements au carré, qui pèse davantage les jours récents), variation du taux japonais à 10 ans en points de base,
  portage (*carry* : écart de taux entre l'euro et le yen), attention et tonalité des news.
- **Standardisation.** z = (x − moyenne) / écart-type, calculés sur l'entraînement seulement, pour que les variables
  soient comparables et que la pénalité traite chaque coefficient de la même façon.

### Modèles (S2 à S5, L1, L2, L3)
- **Régression logistique (S2, L5).** p = σ(w·x + b), avec σ(u) = 1 / (1 + e^(−u)). Elle sort une probabilité ; w·x + b
  est la *log-cote* (log-odds).
- **Log-loss (S2, L5).** −[y log p + (1 − y) log(1 − p)], moyennée sur les mois. C'est l'opposé de la log-vraisemblance :
  minimiser la log-loss revient à maximiser la vraisemblance. Elle punit fort une probabilité confiante et fausse.
- **Descente de gradient (L1).** On part de w = 0 et on avance de −η ∇J à chaque pas. Pour la logistique,
  ∇J = (1/m) Xᵀ(p − y) + 2λw. Le coût est convexe : il n'a qu'un minimum, que la descente atteint si le pas η est
  inférieur à 2/L (L : constante de Lipschitz du gradient). Codé à la main dans `src/logreg_scratch.py`.
- **Régularisation (S3, L7).** On ajoute λ‖w‖² (L2, *ridge*) ou λ‖w‖₁ (L1, *lasso*) au coût. Elle rétrécit les
  coefficients et limite le surapprentissage. Vue bayésienne (L7) : L2 = un *prior* gaussien centré sur 0.
- **Moindres carrés (L2).** Régression linéaire : w = (XᵀX)⁻¹Xᵀy (équations normales). R² = 1 − SCR/SCT ; un R² négatif
  en validation veut dire « pire que la moyenne ».
- **Arbre de décision (S4, L3B).** Coupe l'espace en rectangles en choisissant à chaque nœud la coupe qui réduit le plus
  l'impureté (Gini ou entropie).
- **Forêt aléatoire (S5).** Beaucoup d'arbres, chacun sur un rééchantillon et un sous-ensemble de variables ; on fait la
  moyenne. Réduit la variance.
- **Gradient boosting (S5).** Des arbres peu profonds ajoutés un par un, chacun corrigeant les erreurs des précédents.
  Réduit le biais ; se surajuste vite sans régularisation.
- **k-means (L3C).** Regroupe les mois en k groupes autour de centres ; ici, deux régimes de marché (calme, agité),
  dont la distance sert de variable.

### Réglage et validation (S3, S5, S6)
- **Pipeline scikit-learn.** Enchaîne standardisation, régimes et modèle dans un seul objet, ajusté uniquement sur les
  données d'apprentissage de chaque pli : pas de fuite par le prétraitement.
- **TimeSeriesSplit.** Validation croisée temporelle : chaque pli apprend sur un début de période et valide sur la
  suite.
- **Optuna (TPE).** Cherche les hyperparamètres (λ, profondeur, taux d'apprentissage…) en proposant de nouveaux essais
  là où les précédents étaient bons. Chaque essai est journalisé.

### Classes, seuil, calibration (S7, S8)
- **Déséquilibre et PR-AUC (S7).** Quand une classe est rare, l'exactitude trompe ; la courbe précision-rappel et son
  aire jugent mieux la classe positive. La référence d'une PR-AUC est la fréquence de la classe.
- **Calibration (S8).** Un modèle est calibré si, parmi les mois où il annonce 60 %, la baisse arrive environ 6 fois
  sur 10. **Platt** ajuste une petite logistique sur les scores : p = σ(a·s + c). **Score de Brier** = moyenne de
  (p − y)² ; **ECE** = écart moyen entre probabilité annoncée et fréquence observée, par paquets.
- **Règle de décision (S8).** Agir seulement si l'espérance de gain dépasse le coût : on achète si la probabilité de
  hausse est assez haute, on vend si elle est assez basse, sinon on reste neutre. Les seuils sont choisis sur la
  validation puis **figés** avant le test.

### Explicabilité (S9)
- **Tranches (slices).** La performance par sous-période ou par régime : un score global peut cacher une période où le
  modèle se trompe toujours.
- **Importance par permutation.** On garde le modèle figé, on mélange une variable (ou un groupe de variables
  corrélées) entre les lignes, et on mesure combien la log-loss augmente. Grande hausse = variable utile ; valeur
  négative = le modèle ferait mieux sans elle.
- **SHAP.** Décompose une prédiction : f(x) = φ₀ + Σ φⱼ, sur l'échelle des log-cotes. Pour un modèle linéaire,
  φⱼ = wⱼ(zⱼ − E[zⱼ]). Les valeurs de Shapley répartissent l'écart à la moyenne entre variables, en moyenne sur tous les
  ordres d'arrivée. SHAP décrit le modèle, pas l'économie : aucune causalité.
- **Dépendance partielle (PDP).** La prédiction moyenne quand on fait varier une seule variable.

### Évaluer honnêtement après coup
- **Walk-forward.** Réentraîner chaque mois sur tous les mois passés, prédire le suivant : c'est ce qu'on aurait fait
  en vrai.
- **Test de Diebold-Mariano.** Compare les pertes de deux prévisions mois par mois et dit si l'écart est significatif,
  en tenant compte de l'autocorrélation.
- **Correction de Holm.** Quand on teste beaucoup de variantes, quelques-unes passent par hasard ; Holm resserre les
  seuils de significativité.
- **Bootstrap par blocs.** Rééchantillonne des blocs de mois consécutifs pour obtenir un intervalle de confiance du
  ratio de Sharpe (rendement moyen / volatilité) sans casser la dépendance temporelle.
- **Protocole figé (SHA-256).** Les variantes et critères sont écrits avant le calcul, avec une empreinte ; le code
  refuse de tourner si le protocole a changé. C'est la défense contre le surapprentissage par essais répétés.
- **GARCH, HMM.** GARCH prévoit la volatilité de demain à partir des chocs et de la volatilité d'hier ; un modèle de
  Markov caché (HMM) suppose des régimes cachés (calme, agité) qui se succèdent avec des probabilités de transition.

### Données externes
- **GDELT et FinBERT.** GDELT recense les articles de presse du monde entier ; FinBERT, un modèle de langue entraîné sur
  des textes financiers, note leur tonalité (positive, négative, neutre).
- **Bloomberg.** Référence pour la seconde version : il sert à contrôler les séries (Tankan, IPC, VIX) et leurs dates
  de publication. Les exports bruts restent hors de Git (licence).

## 1 · Configuration

Un seul endroit pour les dates du découpage, les listes de variables et la graine aléatoire : `config.yaml` (première
version, 2010-2026) et `config_bloomberg.yaml` (seconde version, 1999-2026). `src/config.py` choisit l'un ou l'autre
selon la variable d'environnement `NIKKEI_CONFIG` et note le commit courant dans chaque résultat, pour la traçabilité.

In [2]:
montrer("src/config.py")

**`src/config.py`** · 26 lignes

"""Chemins et configuration communs à tous les modules."""
import os
from pathlib import Path
import subprocess

import yaml

ROOT = Path(__file__).resolve().parents[1]
# NIKKEI_CONFIG choisit une autre configuration (config_bloomberg.yaml : seconde version, après le test)
CONFIG = ROOT / os.environ.get("NIKKEI_CONFIG", "config.yaml")
CFG = yaml.safe_load(open(CONFIG))
SEED = CFG["seed"]
RUNS = ROOT / CFG.get("runs", "runs")
MODELS = RUNS / "models"
FIG = ROOT / "figures"

FEATURES = [c for groupe in ["news", "price", "macro", "fx"] for c in CFG["features"][groupe]]
REGIME = CFG["features"]["regime"]
COLONNES = FEATURES + [c for c in REGIME if c not in FEATURES]
CIBLES = {"nikkei": "y_baisse", "couverture": "y_non_couvert"}


def commit_courant() -> str:
    """Hash court du commit Git en cours (pour le journal des essais)."""
    sortie = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True)
    return sortie.stdout.strip() or "inconnu"

In [3]:
import yaml
for nom in ["config.yaml", "config_bloomberg.yaml"]:
    c = yaml.safe_load((RACINE / nom).read_text())
    print(nom, "→", c["split"])
    print("   variables :", [v for k in ["news", "price", "macro", "fx"] for v in c["features"][k]])

config.yaml → {'start': '2010-02-01', 'train_end': '2021-12-31', 'validation_end': '2023-12-31', 'test_end': '2026-09-30'}
   variables : ['news_attention_6m', 'news_sent_w', 'news_dispo', 'n225_mom_12m', 'n225_mom_3m', 'n225_vol_ewma', 'd_jgb10y_bp', 'tankan_large_manuf', 'carry_1m', 'eurjpy_rvol_1m']
config_bloomberg.yaml → {'start': '1999-01-01', 'train_end': '2021-12-31', 'validation_end': '2023-12-31', 'test_end': '2026-09-30'}
   variables : ['news_attention_6m', 'news_sent_w', 'news_dispo', 'n225_mom_12m', 'n225_mom_3m', 'n225_vol_ewma', 'd_jgb10y_bp', 'tankan_large_manuf', 'carry_1m', 'eurjpy_rvol_1m']


## 1 · Données : de la table mensuelle à la table de modélisation

**Ce que fait le fichier.** Il lit la table mensuelle construite par les notebooks de collecte, ajoute les cibles du
mois suivant, construit les variables de news, attribue à chaque mois son bloc (entraînement, validation, test) et
lance l'**audit des fuites**, qui arrête tout si un contrôle échoue : la cible est bien le rendement du mois suivant,
aucune cible n'est parmi les variables, le Tankan ne change qu'aux mois de publication (janvier, avril, juillet,
octobre), les variables de news calculées sur des données coupées en 2020 sont identiques (elles ne regardent pas le
futur), et aucune variable n'est presque parfaitement corrélée (Spearman) à une cible.

**Concepts.** Fuite d'information, décalage de publication, cible décalée d'un mois (`shift(-1)`), découpage
chronologique (S1).

In [4]:
montrer("src/data.py")

**`src/data.py`** · 106 lignes

"""Étape A : de la table mensuelle à la table de modélisation (notebook 01).

Entrée : data/2_construites/dataset_monthly.csv, construit par les notebooks de collecte 00a et 00b
(téléchargement, nettoyage, calendrier de Tokyo, décalages de publication, agrégation mensuelle).
Sortie : data/3_modele/dataset_model.csv, avec les cibles, les variables construites et le bloc du split.
"""
import numpy as np
import pandas as pd

from .config import CFG, COLONNES, FEATURES, REGIME, ROOT

GARDER_EN_PLUS = ["y_baisse", "y_non_couvert", "target_up_next", "target_rn_next", "target_fx_next",
                  "target_reur_unhedged_next", "target_reur_hedged_next", "target_short_eur_next",
                  "eur_overnight", "boj_meeting_next_month", "n225_close", "bloc"]


def charger_mensuel() -> pd.DataFrame:
    return pd.read_csv(ROOT / CFG["data"]["monthly"], parse_dates=["date"]).set_index("date")


def ajouter_cibles(df: pd.DataFrame) -> pd.DataFrame:
    """y_baisse = 1 si le Nikkei baisse le mois suivant ; y_non_couvert = 1 si ne pas couvrir était mieux."""
    df = df.copy()
    inconnu = df["target_rn_next"].isna()
    df["y_baisse"] = (df["target_rn_next"] <= 0).astype(float)
    df["y_non_couvert"] = (df["carry_1m"] - df["target_fx_next"] <= 0).astype(float)
    df.loc[inconnu, ["y_baisse", "y_non_couvert"]] = np.nan
    return df


def variables_news(df: pd.DataFrame) -> pd.DataFrame:
    """Volume anormal d'articles, sentiment pondéré et indicateur de présence des news.

    Chaque ligne n'utilise que son propre passé : le calcul peut se faire avant le split.
    """
    n_tot = (df["news_japan_n_1m"] + df["news_boj_n_1m"]).where(lambda n: n > 0)
    log_n = np.log(n_tot)
    out = pd.DataFrame(index=df.index)
    out["news_attention_6m"] = log_n - log_n.shift(1).rolling(6, min_periods=3).mean()
    out["news_sent_w"] = (df["news_japan_n_1m"] * df["news_japan_sent_1m"]
                          + df["news_boj_n_1m"] * df["news_boj_sent_1m"]) / n_tot
    out["news_dispo"] = out["news_attention_6m"].notna().astype(int)
    return out


def ajouter_blocs(df: pd.DataFrame) -> pd.DataFrame:
    s = {k: pd.Timestamp(v) for k, v in CFG["split"].items()}
    df = df.copy()
    df["bloc"] = np.select(
        [df.index < s["start"], df.index <= s["train_end"], df.index <= s["validation_end"], df.index <= s["test_end"]],
        ["hors période", "entraînement", "validation", "test"], default="à prédire")
    return df


def audit_fuites(df: pd.DataFrame) -> None:
    """Contrôles du notebook 01 : lèvent une erreur si une fuite est détectée."""
    ecart_cible = (df["target_rn_next"] - df["n225_ret_1m"].shift(-1)).abs().max()
    assert ecart_cible < 1e-9, "la cible n'est pas le rendement du mois suivant"
    ecart_couv = ((df["target_reur_hedged_next"] - df["target_reur_unhedged_next"])
                  - (df["carry_1m"] - df["target_fx_next"])).abs().max()
    assert ecart_couv < 1e-5, "l'identité de la couverture ne tient pas"
    cibles = {c for c in df.columns if c.startswith("target_")} | {"y_baisse", "y_non_couvert"}
    assert not cibles & set(FEATURES + REGIME), "une cible est parmi les variables"
    mois_tankan = set(df.index[df["tankan_large_manuf"].diff().fillna(0) != 0].month)
    assert mois_tankan <= {1, 4, 7, 10}, "le Tankan change avant sa publication"
    coupe = variables_news(df.loc[:"2020-06-30"])
    assert np.allclose(variables_news(df).loc[:"2020-06-30"].to_numpy(), coupe.to_numpy(), equal_nan=True), \
        "les variables de news regardent le futur"
    ecart_prix = (df["target_rn_next"] - (df["n225_close"].shift(-1) / df["n225_close"] - 1)).abs().max()
    assert ecart_prix < 1e-5, "la cible ne correspond pas aux prix de t et t+1"
    # Contrôle sur les valeurs, pas seulement sur les noms : aucune variable ne doit recopier une cible
    connus = df["target_rn_next"].notna()
    for col in FEATURES + REGIME:
        for cible in ["target_rn_next", "target_fx_next", "y_baisse"]:
            paire = df.loc[con

## 2 · La pipeline et les espaces de recherche

**Ce que fait le fichier.** `construire_pipeline` assemble : imputation et standardisation, ajout des distances aux
régimes k-means (calme, agité), puis le modèle. `proposer` décrit pour Optuna l'espace de chaque modèle : force de la
régularisation de la logistique, nombre et profondeur des arbres, taux d'apprentissage du boosting.

**Concepts.** Pipeline sans fuite (le prétraitement s'ajuste dans chaque pli), standardisation, k-means (L3C),
logistique régularisée (S2, S3), forêt et boosting (S4, S5).

In [5]:
montrer("src/pipeline.py")

**`src/pipeline.py`** · 50 lignes

"""Étape B : la pipeline scikit-learn et les espaces de recherche des modèles (notebook 02)."""
from lightgbm import LGBMClassifier
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .config import FEATURES, REGIME, SEED

K_REGIMES = 2   # coude du k-means sur l'entraînement (notebook 02, L3B)


def construire_pipeline(modele, k: int = K_REGIMES) -> Pipeline:
    """Imputation, distances aux régimes k-means, standardisation, puis le modèle.

    Tout ce qui apprend un paramètre est dans la pipeline, donc ajusté sur le fold d'entraînement seulement.
    """
    regime = Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler()),
                       ("kmeans", KMeans(n_clusters=k, n_init=10, random_state=SEED))])
    prep = ColumnTransformer([("vars", SimpleImputer(strategy="median"), FEATURES),
                              ("regime", regime, REGIME)])
    return Pipeline([("prep", prep), ("scale", StandardScaler()), ("model", modele)])


def proposer(trial, nom: str):
    """Modèle proposé par un essai Optuna (ou par optuna.trial.FixedTrial pour des paramètres fixés)."""
    poids = trial.suggest_categorical("class_weight", [None, "balanced"])
    if nom == "logit":
        # saga ne pénalise pas l'intercept b (liblinear le pénalise)
        return LogisticRegression(C=trial.suggest_float("C", 1e-3, 10, log=True),
                                  penalty=trial.suggest_categorical("penalty", ["l1", "l2"]),
                                  solver="saga", class_weight=poids, max_iter=5000, random_state=SEED)
    if nom == "rf":
        return RandomForestClassifier(n_estimators=trial.suggest_int("n_estimators", 100, 500, step=100),
                                      max_depth=trial.suggest_int("max_depth", 2, 4),
                                      min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 15),
                                      max_features=trial.suggest_categorical("max_features", ["sqrt", 0.5]),
                                      class_weight=poids, random_state=SEED, n_jobs=1)
    if nom == "lgbm":
        return LGBMClassifier(n_estimators=trial.suggest_int("n_estimators", 50, 300, step=25),
                              learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                              num_leaves=trial.suggest_int("num_leaves", 4, 8),
                              min_child_samples=trial.suggest_int("min_child_samples", 5, 15),
                              reg_lambda=trial.suggest_float("reg_lambda", 0.0, 5.0),
                              class_weight=poids, random_state=SEED, n_jobs=1, verbose=-1)
    raise ValueError(f"modèle inconnu : {nom}")

## 2 · Réglage Optuna

**Ce que fait le fichier.** Pour chaque modèle, Optuna propose des hyperparamètres ; chaque proposition est jugée par
la log-loss moyenne sur les plis d'un `TimeSeriesSplit` de l'entraînement. Tous les essais sont écrits dans un journal ;
le meilleur de chaque modèle est réajusté et sauvegardé.

**Concepts.** Validation croisée temporelle, recherche bayésienne TPE, log-loss comme critère (S5, S6).

In [6]:
montrer("src/tune.py")

**`src/tune.py`** · 104 lignes

"""Réglage Optuna avec TimeSeriesSplit, journal des essais, modèles retenus (notebook 02).

Usage : python -m src.tune [--essais 50] [--sortie runs]
"""
import argparse
import json
import time
import warnings
from datetime import datetime

import joblib
import numpy as np
import optuna
import pandas as pd
from optuna.samplers import TPESampler
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss
from sklearn.model_selection import TimeSeriesSplit

from .config import CIBLES, COLONNES, RUNS, SEED, commit_courant
from .data import charger_table
from .pipeline import construire_pipeline, proposer

PLAN = [("nikkei", "logit"), ("nikkei", "rf"), ("nikkei", "lgbm"), ("couverture", "logit")]
FOLDS_CV = TimeSeriesSplit(n_splits=4, test_size=12)


def evaluer_cv(pipe, X, y, folds):
    """Log-loss, Brier et PR-AUC moyens sur les folds, et prédictions hors fold."""
    pertes, briers, praucs = [], [], []
    oof = pd.Series(np.nan, index=y.index)
    for a, b in folds:
        modele = clone(pipe).fit(X.iloc[a], y.iloc[a])
        p = modele.predict_proba(X.iloc[b])[:, 1]
        pertes.append(log_loss(y.iloc[b], p, labels=[0, 1]))
        briers.append(brier_score_loss(y.iloc[b], p))
        praucs.append(average_precision_score(y.iloc[b], p))
        oof.iloc[b] = p
    return {"logloss": float(np.mean(pertes)), "logloss_std": float(np.std(pertes)),
            "brier": float(np.mean(briers)), "pr_auc": float(np.mean(praucs)),
            "logloss_folds": [round(float(x), 4) for x in pertes]}, oof


def main(n_essais: int = 50, sortie=RUNS) -> None:
    from pathlib import Path
    sortie = Path(sortie)
    modeles = sortie / "models"
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", message=".*feature names.*")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    modeles.mkdir(parents=True, exist_ok=True)

    table = charger_table()
    train, valid = table[table["bloc"] == "entraînement"], table[table["bloc"] == "validation"]
    X_tr, X_va = train[COLONNES], valid[COLONNES]
    folds = list(FOLDS_CV.split(X_tr))
    date, commit = datetime.now().strftime("%Y-%m-%d %H:%M"), commit_courant()

    journal, predictions = [], []
    for cible, nom in PLAN:
        y_tr, y_va = train[CIBLES[cible]].astype(int), valid[CIBLES[cible]].astype(int)

        def objectif(trial):
            res, _ = evaluer_cv(construire_pipeline(proposer(trial, nom)), X_tr, y_tr, folds)
            for cle in ["logloss_std", "brier", "pr_auc", "logloss_folds"]:
                trial.set_user_attr(cle, res[cle])
            return res["logloss"]

        debut = time.time()
        etude = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
        etude.optimize(objectif, n_trials=n_essais)
        meilleur = etude.best_trial

        fixe = lambda: construire_pipeline(proposer(optuna.trial.FixedTrial(etude.best_params), nom))
        pipe = fixe().fit(X_tr, y_tr)
        joblib.dump(pipe, modeles / f"{cible}_{nom}.joblib")
        _, oof = evaluer_cv(fixe(), X_tr, y_tr, folds)
        predictions.append(pd.DataFrame({"cible": cible, "modele": nom, "bloc": "hors fold", "y": y_tr, "p": oof}).dropna())
        predictions.append(pd.DataFrame({"cible": cible, "modele": nom, "bloc": "validation", "y": y_va,
                                         "p": pipe.predict_proba(X_va)[:, 1]}, index=valid.index))

        for t in etude.trials:
            retenu = t.number == meilleur.number
            journal.append({"date": date, "commit": commit, "cible": cible, "modele": nom, "essai": t.number,
                            "params": json.dumps(t.params), "cv_logloss": round(t.value, 5),
                            "cv_logloss_std": round(t.user_attrs["logloss_std"], 5),
                            "cv_brier": round(t.user_attrs["brier"], 5), "cv_pr_auc": round(t.user_attrs["pr_

## 2 · La logistique à la main (annexe de Maths)

**Ce que fait le fichier.** Il réécrit la régression logistique pénalisée sans scikit-learn : la sigmoïde, le coût
J_λ, son gradient, la constante de Lipschitz qui borne le pas, et la descente de gradient. Le notebook 05 vérifie
qu'on retrouve les coefficients de scikit-learn.

**Concepts.** Vraisemblance et log-loss (L5), gradient et descente (L1), pénalité L2 et prior gaussien (L7),
convexité : un seul minimum (L7, L8).

In [7]:
montrer("src/logreg_scratch.py")

**`src/logreg_scratch.py`** · 42 lignes

"""Régression logistique L2 codée à la main, par descente de gradient (annexe de Maths, notebook 05).

J_λ(w, b) = -(1/m) Σ [y log p + (1 - y) log(1 - p)] + λ ||w||²,   p = σ(w·x + b)
∇_w J_λ = (1/m) Xᵀ(p - y) + 2λ w,   ∂J_λ/∂b = (1/m) Σ (p - y)   (b n'est pas pénalisé)
"""
import numpy as np


def sigmoide(u):
    return 1 / (1 + np.exp(-u))


def cout(w, b, X, y, lam=0.0):
    p = np.clip(sigmoide(X @ w + b), 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)) + lam * (w @ w)


def gradient(w, b, X, y, lam=0.0):
    r = sigmoide(X @ w + b) - y
    return X.T @ r / len(y) + 2 * lam * w, r.mean()


def constante_lipschitz(X, lam):
    """Borne L ≤ ¼ λmax(X̃ᵀX̃/m) + 2λ, avec X̃ = [X, 1] ; la descente converge pour α < 2/L."""
    Xt = np.c_[X, np.ones(len(X))]
    return 0.25 * np.linalg.eigvalsh(Xt.T @ Xt / len(X)).max() + 2 * lam


def descente(X, y, alpha, lam, T, w=None, b=0.0):
    """Descente de gradient ; renvoie les poids, l'intercept et l'historique du coût."""
    w = np.zeros(X.shape[1]) if w is None else np.asarray(w, dtype=float).copy()
    historique = np.empty(T)
    for t in range(T):
        gw, gb = gradient(w, b, X, y, lam)
        w, b = w - alpha * gw, b - alpha * gb
        historique[t] = cout(w, b, X, y, lam)
    return w, b, historique


def ajuster(X, y, lam, T=20000):
    """Ajuste avec le pas classique α = 1/L."""
    return descente(X, y, 1 / constante_lipschitz(X, lam), lam, T)

## 2 · Calibration, règle de décision, décision figée

**Ce que fait le fichier.** Il calibre les scores par Platt sur la validation, calcule les rendements des actions
(acheter, neutre, vendre ; couvert ou non), choisit les seuils qui maximisent l'espérance de gain, et **fige** le tout
dans `runs/decision_finale.json`. `--verifier` recalcule et compare au fichier sans rien écrire : la décision ne peut
pas être modifiée en douce après le test.

**Concepts.** Calibration de Platt, espérance de gain et coût de transaction, seuils, engagement avant le test (S8).

In [8]:
montrer("src/decision.py")

**`src/decision.py`** · 116 lignes

"""Calibration de Platt, règle de décision par espérance de gain, et décision figée (notebook 03).

La décision du projet est figée dans runs/decision_finale.json (commit 0d8df2f, avant le test).
Usage :
  python -m src.decision --verifier   recalcule tout et compare à la décision figée, sans rien écrire
  python -m src.decision --ecrire     réécrit le fichier (refusé s'il existe déjà, sauf --forcer)
"""
import argparse
import json
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from .config import RUNS, commit_courant
from .data import charger_table

MODELES = ["logit", "rf", "lgbm"]
MARGE = 0.005          # gain espéré minimal au-dessus du cash pour prendre position, fixé à l'avance
FICHIER = RUNS / "decision_finale.json"


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def apprendre_platt(p_hors_fold, y_hors_fold) -> dict:
    """Platt : sigmoïde du log-odds du modèle, apprise sur les prédictions hors fold."""
    cal = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p_hors_fold).reshape(-1, 1), y_hors_fold)
    return {"a": float(cal.coef_[0, 0]), "c": float(cal.intercept_[0])}


def appliquer_platt(p, platt: dict):
    return 1 / (1 + np.exp(-(platt["a"] * logit(p) + platt["c"])))


def couts_entrainement(train: pd.DataFrame) -> dict:
    """Rendements moyens en excès du cash d'un achat couvert et d'une vente, selon hausse ou baisse."""
    cash = train["eur_overnight"] / 100 / 12
    achat = train["target_reur_hedged_next"] - cash
    vente = train["target_short_eur_next"] - cash
    hausse = train["y_baisse"] == 0
    return {"B_u": float(achat[hausse].mean()), "B_d": float(achat[~hausse].mean()),
            "S_u": float(vente[hausse].mean()), "S_d": float(vente[~hausse].mean())}


def seuils(couts: dict, marge: float) -> dict:
    return {"theta_a": (marge - couts["B_d"]) / (couts["B_u"] - couts["B_d"]),
            "theta_v": (couts["S_d"] - marge) / (couts["S_d"] - couts["S_u"])}


def actions(p_hausse: pd.Series, couts: dict, marge: float) -> pd.Series:
    """Acheter ou vendre si le gain espéré au-dessus du cash dépasse la marge, sinon neutre."""
    gain_achat = p_hausse * couts["B_u"] + (1 - p_hausse) * couts["B_d"]
    gain_vente = p_hausse * couts["S_u"] + (1 - p_hausse) * couts["S_d"]
    meilleur = np.where(gain_achat >= gain_vente, "acheter", "vendre")
    return pd.Series(np.where(np.maximum(gain_achat, gain_vente) >= marge, meilleur, "neutre"), index=p_hausse.index)


def rendements(act: pd.Series, bloc: pd.DataFrame) -> pd.Series:
    """Rendement en euros du mois suivant : achat couvert, cash ou vente."""
    cash = bloc["eur_overnight"] / 100 / 12
    return pd.Series(np.select([act == "acheter", act == "vendre"],
                               [bloc["target_reur_hedged_next"], bloc["target_short_eur_next"]], cash), index=bloc.index)


def calculer_decision() -> dict:
    table = charger_table()
    train = table[table["bloc"] == "entraînement"]
    pred = pd.read_csv(RUNS / "predictions_02.csv", parse_dates=["date"])
    hors_fold = pred[(pred["cible"] == "nikkei") & (pred["bloc"] == "hors fold")]
    platt = {}
    for m in MODELES:
        d = hors_fold[hors_fold["modele"] == m]
        platt[m] = apprendre_platt(d["p"].to_numpy(), d["y"].astype(int).to_numpy())
    couts = couts_entrainement(train)
    return {"modele_nikkei": "logit", "platt": platt, "couts_entrainement_exces_du_cash": couts,
            "marge_m": MARGE, "seuils": seuils(couts, MARGE), "couverture": "toujours couvert (h = 1)"}


def charger_decision() -> dict:
    return json.loads(FICHIER.read_text())


def verifier() -> None:
    """Recalcule la décision et la compare au fichier figé."""
    fige, recalcul = charger_decision(), calculer_decision()
    ecarts = {"platt " + m: max(abs(fige["platt"][m][k] - recalcul["platt"][m][k]) for k in "ac") for m in MODELES}
    ecarts["coûts"] = max(abs(f

In [9]:
d = lire_json("runs/decision_finale.json")
print({k: d[k] for k in list(d)[:8]})

{'date': '2026-11-11 18:08', 'commit_du_code': '2aa4b40', 'modele_nikkei': 'logit', 'fichiers_modeles': {'logit': 'runs/models/nikkei_logit.joblib', 'rf': 'runs/models/nikkei_rf.joblib', 'lgbm': 'runs/models/nikkei_lgbm.joblib'}, 'sortie_du_modele': 'P(baisse) ; p_hausse = 1 - P(baisse) calibrée', 'platt': {'logit': {'a': 0.9154481615258249, 'c': 0.01041380261161088}, 'rf': {'a': 1.5359887472351588, 'c': 0.1324221642348835}, 'lgbm': {'a': 0.9948037455190625, 'c': -0.10098822029913068}}, 'couts_entrainement_exces_du_cash': {'B_u': 0.038925706206896546, 'B_d': -0.04178659265476191, 'S_u': -0.038852560137931036, 'S_d': 0.04182417268452381}, 'platt_appris_sur': 'prédictions hors fold 2018-2021 (TimeSeriesSplit du notebook 02)'}


## 2 · Le test, ouvert une seule fois

**Ce que fait le fichier.** Il applique la décision figée aux mois 2024-2026 et mesure log-loss, Brier, PR-AUC et
rendement en euros, contre des références (probabilité constante, toujours acheter). Le résultat est dans
`runs/evaluation_test.json` ; le fichier ne sert plus qu'à vérifier qu'on retrouve les mêmes chiffres.

**Concepts.** Test unique, références naïves, métriques probabilistes et financières (S1, S7, S8).

In [10]:
montrer("src/evaluate.py")

**`src/evaluate.py`** · 53 lignes

"""Évaluation sur le test 2024-2026 avec la décision figée (notebook 03b).

Le test a été évalué une seule fois, au commit 775a352 (runs/evaluation_test.json).
Usage : python -m src.evaluate --verifier   recalcule et compare au fichier, sans rien écrire
"""
import argparse
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss

from .config import COLONNES, ROOT, RUNS
from .data import charger_table
from .decision import actions, appliquer_platt, charger_decision, rendements

FICHIER = RUNS / "evaluation_test.json"


def evaluer() -> dict:
    table, dec = charger_table(), charger_decision()
    train, test = table[table["bloc"] == "entraînement"], table[table["bloc"] == "test"]
    y = test["y_baisse"].astype(int)
    pipe = joblib.load(ROOT / dec["fichiers_modeles"][dec["modele_nikkei"]])
    p_baisse = appliquer_platt(pipe.predict_proba(test[COLONNES])[:, 1], dec["platt"][dec["modele_nikkei"]])
    p_const = np.full(len(y), train["y_baisse"].mean())
    act = actions(pd.Series(1 - p_baisse, index=test.index), dec["couts_entrainement_exces_du_cash"], dec["marge_m"])
    cumul = lambda r: float(np.prod(1 + np.asarray(r)) - 1)
    return {"log_loss_modele": log_loss(y, p_baisse), "log_loss_constante": log_loss(y, p_const),
            "brier_modele": brier_score_loss(y, p_baisse), "pr_auc_modele": average_precision_score(y, p_baisse),
            "actions": act.value_counts().to_dict(), "cumul_regle": cumul(rendements(act, test)),
            "cumul_achat_couvert": cumul(test["target_reur_hedged_next"])}


def verifier() -> None:
    fige, r = json.loads(FICHIER.read_text()), evaluer()
    attendu = {"log_loss_modele": fige["metriques"]["logistique"]["log-loss"],
               "log_loss_constante": fige["metriques"]["constante (fréquence de l'entraînement)"]["log-loss"],
               "cumul_regle": fige["performance_euros"]["règle du modèle"]["cumul"],
               "cumul_achat_couvert": fige["performance_euros"]["toujours acheter, couvert"]["cumul"]}
    for cle, valeur in attendu.items():
        print(f"{cle:22s} figé {valeur:.4f} | recalculé {r[cle]:.4f}")
        assert abs(valeur - r[cle]) < 1e-3, f"{cle} ne correspond pas"
    assert r["actions"] == fige["actions"], "les actions ne correspondent pas"
    print("évaluation du test retrouvée à l'identique (rien n'a été réécrit)")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--verifier", action="store_true", required=True)
    parser.parse_args()
    verifier()

## 3 · Signal, historique et explication SHAP

**Ce que fait le fichier.** Il recharge le modèle figé, calcule la probabilité du dernier mois connu, l'action de la
règle et la **contribution SHAP** de chaque variable (pour la logistique : coefficient × écart à la moyenne de
l'entraînement, sur l'échelle des log-cotes). C'est ce que lisent l'API, le tableau de bord et le site.

**Concepts.** SHAP linéaire, additivité f(x) = φ₀ + Σφⱼ, lecture en cascade (*waterfall*) (S9). L'importance par
permutation et les tranches sont dans le notebook 04.

In [11]:
montrer("src/predict.py")

**`src/predict.py`** · 93 lignes

"""Signal du mois, historique des décisions et explication SHAP, avec la décision figée.

Utilisé par la plateforme (API et tableau de bord). Usage : python -m src.predict [--mois 2026-10]
"""
import argparse
from functools import lru_cache

import joblib
import numpy as np
import pandas as pd

from .config import COLONNES, FEATURES, ROOT
from .data import charger_table
from .decision import actions, appliquer_platt, charger_decision, rendements

RECOMMANDATION = "toujours acheter, couvert (le modèle ne bat pas la constante sur le test)"


@lru_cache(maxsize=1)
def _modele():
    dec = charger_decision()
    return dec, joblib.load(ROOT / dec["fichiers_modeles"][dec["modele_nikkei"]])


def _ligne(table: pd.DataFrame, mois: str | None) -> pd.DataFrame:
    ligne = table[table["bloc"] == "à prédire"] if mois is None else table[table.index.strftime("%Y-%m") == mois]
    if ligne.empty:
        raise KeyError(f"aucune donnée pour le mois {mois}")
    return ligne


def probabilites_hausse(lignes: pd.DataFrame) -> pd.Series:
    """Probabilité calibrée de hausse le mois suivant : 1 - Platt(P(baisse))."""
    dec, pipe = _modele()
    p_baisse = appliquer_platt(pipe.predict_proba(lignes[COLONNES])[:, 1], dec["platt"][dec["modele_nikkei"]])
    return pd.Series(1 - p_baisse, index=lignes.index)


def signal(mois: str | None = None, table: pd.DataFrame | None = None) -> dict:
    """Signal pour le mois suivant, à partir des données de fin de `mois` (par défaut, la ligne à prédire)."""
    table = charger_table() if table is None else table
    dec, _ = _modele()
    ligne = _ligne(table, mois)
    p_hausse = float(probabilites_hausse(ligne).iloc[0])
    action = actions(pd.Series([p_hausse], index=ligne.index), dec["couts_entrainement_exces_du_cash"], dec["marge_m"]).iloc[0]
    resultat = ligne["target_rn_next"].iloc[0]
    return {"mois_des_donnees": ligne.index[0].strftime("%Y-%m"), "bloc": ligne["bloc"].iloc[0],
            "p_hausse": round(p_hausse, 4), "score": round(2 * p_hausse - 1, 4),
            "action_du_modele": action, "couverture": "couvert" if action == "acheter" else "sans objet",
            "seuils": {k: round(v, 4) for k, v in dec["seuils"].items()},
            "recommandation_finale": RECOMMANDATION,
            "rendement_realise_mois_suivant": None if np.isnan(resultat) else round(float(resultat), 4)}


def historique(table: pd.DataFrame | None = None, blocs=("validation", "test")) -> pd.DataFrame:
    """Probabilité, action et rendements en euros mois par mois, pour les blocs demandés."""
    table = charger_table() if table is None else table
    dec, _ = _modele()
    d = table[table["bloc"].isin(blocs)]
    p = probabilites_hausse(d)
    act = actions(p, dec["couts_entrainement_exces_du_cash"], dec["marge_m"])
    return pd.DataFrame({"bloc": d["bloc"], "p_hausse": p, "action": act,
                         "hausse_realisee": (d["y_baisse"] == 0).astype(int),
                         "rendement_regle": rendements(act, d),
                         "rendement_achat_couvert": d["target_reur_hedged_next"],
                         "rendement_achat_non_couvert": d["target_reur_unhedged_next"],
                         "rendement_vente": d["target_short_eur_next"],
                         "rendement_cash": d["eur_overnight"] / 100 / 12})


def contributions(mois: str | None = None, table: pd.DataFrame | None = None) -> pd.DataFrame:
    """SHAP du modèle linéaire : φ_j = w_j (z_j - moyenne d'entraînement de z_j), en log-odds de la baisse."""
    table = charger_table() if table is None else table
    _, pipe = _modele()
    prep, w = pipe[:-1], pipe.named_steps["model"].coef_[0]
    z_moyen = prep.transform(table.loc[table["bloc"] == "entraînement", COLONNES]).mean(axis=0)
    ligne = _ligne(table, mois)
    z = prep.transform(ligne[COLONNES])[0]
    regime = pipe.named_steps["prep"].named_transformers_["regime"]
    vix_des_centres = regime.named_steps["scale"].inverse_transform(regime.named_steps["kmeans"].

## 3 · Équité entre régimes et mitigation

**Ce que fait le fichier.** Il mesure, sur le test, le recall, le taux de faux positifs et la part de mois prédits en
baisse dans chaque régime k-means (calme, agité), puis réentraîne la même pipeline avec la repondération de Kamiran et
Calders, w(g, y) = P(g) P(y) / P(g, y), qui rend régime et cible indépendants à l'entraînement, et compare avant/après.
Il tourne sur la version 1 et sur la version Bloomberg (`NIKKEI_CONFIG=config_bloomberg.yaml`).

**Concepts.** Égalité des chances, égalité des cotes, parité démographique, mitigation par prétraitement (exigence 10).

In [12]:
montrer("src/equite.py")

**`src/equite.py`** · 130 lignes

"""Exigence 10 : audit d'équité entre régimes de marché, et une mitigation par repondération (Kamiran et Calders).

Le jeu ne contient pas de personnes. L'attribut « sensible » retenu est le régime du mois (calme ou agité), donné par
le k-means de la pipeline ajusté sur l'entraînement : un modèle qui raterait surtout les baisses des mois agités
ferait perdre de l'argent au pire moment.

Définitions d'équité, avec la baisse comme classe positive et le seuil 0,5 :
  - égalité des chances (equal opportunity) : même recall dans chaque groupe ;
  - égalité des cotes (equalized odds) : même recall et même taux de faux positifs ;
  - parité démographique : même taux de prédictions « baisse ».

Mitigation (prétraitement) : on réentraîne la même pipeline, avec les mêmes hyperparamètres, en pondérant chaque mois
d'entraînement par w(g, y) = P(g) P(y) / P(g, y), qui rend le régime et la cible indépendants dans les données
d'apprentissage. La calibration de Platt est réapprise de la même façon que pour le modèle figé (prédictions hors
fold d'un TimeSeriesSplit de l'entraînement). Rien n'est réglé sur le test.

Ce script est écrit après l'ouverture du test : la décision et l'évaluation figées de runs/ ne sont pas touchées.
Usage : python -m src.equite   (écrit runs/equite_mitigation.json)
       NIKKEI_CONFIG=config_bloomberg.yaml python -m src.equite   (seconde version, runs/v2_bloomberg/)
"""
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.model_selection import TimeSeriesSplit

from src.config import CFG, ROOT, commit_courant
from src.decision import apprendre_platt, appliquer_platt

RUNS = ROOT / CFG.get("runs", "runs")   # runs/ (version 1) ou runs/v2_bloomberg (NIKKEI_CONFIG=config_bloomberg.yaml)
DEC = json.loads((RUNS / "decision_finale.json").read_text())
FEATURES = [c for g in ["news", "price", "macro", "fx"] for c in CFG["features"][g]]
REGIME = CFG["features"]["regime"]
COLONNES = FEATURES + [c for c in REGIME if c not in FEATURES]
SEUIL = 0.5


def charger():
    table = pd.read_csv(ROOT / CFG["data"]["model_table"], parse_dates=["date"]).set_index("date")
    pipe = joblib.load(ROOT / DEC["fichiers_modeles"]["logit"])
    return table, pipe


def regimes(pipe, X) -> pd.Series:
    """Régime k-means de chaque mois, nommé d'après le centre au VIX le plus haut."""
    km = pipe.named_steps["prep"].named_transformers_["regime"]
    centres = km.named_steps["scale"].inverse_transform(km.named_steps["kmeans"].cluster_centers_)
    agite = int(np.argmax(centres[:, REGIME.index("vix")]))
    return pd.Series(km.predict(X[REGIME]), index=X.index).map({agite: "agité", 1 - agite: "calme"})


def poids_kamiran(groupe: pd.Series, y: pd.Series) -> np.ndarray:
    """w(g, y) = P(g) P(y) / P(g, y) : le groupe et la cible deviennent indépendants une fois pondérés."""
    pg, py = groupe.value_counts(normalize=True), y.value_counts(normalize=True)
    pgy = pd.crosstab(groupe, y, normalize=True)
    return np.array([pg[g] * py[c] / pgy.loc[g, c] for g, c in zip(groupe, y)])


def p_platt(pipe, X, platt):
    return appliquer_platt(pipe.predict_proba(X[COLONNES])[:, 1], platt)


def platt_hors_fold(pipe, X, y, w):
    """Comme le modèle figé : Platt appris sur les prédictions hors fold de l'entraînement."""
    p, cible = [], []
    for app, val in TimeSeriesSplit(n_splits=4, test_size=12).split(X):
        m = clone(pipe).fit(X.iloc[app][COLONNES], y.iloc[app], model__sample_weight=w[app])
        p.append(m.predict_proba(X.iloc[val][COLONNES])[:, 1])
        cible.append(y.iloc[val])
    return apprendre_platt(np.concatenate(p), np.concatenate(cible))


def mesures(p, y, groupe) -> dict:
    yhat = (p >= SEUIL).astype(int)
    par = {}
    for g in ["calme", "agité"]:
        m = (groupe == g).to_numpy()
        pos, neg = (y[m] == 1), (y[m] == 0)
        par[g] = {"mois": int(m.sum()), "baisses": int(pos.sum()),
              

## 4 · Seconde version : 1999-2026, Bloomberg en référence

**Ce que fait le fichier.** Après l'ouverture du test, il rejoue tout le protocole sur une histoire plus longue :
1. **Contrôle Bloomberg** : compare le Tankan, l'IPC et le VIX de la table aux exports Bloomberg. Le Tankan coïncide
   une fois décalé d'un mois (sa publication) ; pris à sa date Bloomberg il aurait fui. L'export du VIX, converti en
   euros par erreur, est écarté.
2. Réglage Optuna sur 1999-2021, calibration, seuils, puis évaluation et SHAP sur le test.

Tout s'écrit dans `runs/v2_bloomberg/` : la décision et l'évaluation de la première version ne sont pas touchées. C'est
cette table (`data/3_modele/dataset_model2.csv`) et ces contrôles que le site affiche dans la salle « Le dataset » et
le chapitre « Après coup ».

**Concepts.** Contrôle de qualité des données, dates de publication, réentraînement sur une histoire plus longue.

In [13]:
montrer("src/v2_bloomberg.py")

**`src/v2_bloomberg.py`** · 226 lignes

"""Seconde version, après l'ouverture du test : données 1999-2026 et séries Bloomberg en référence.

Même protocole que la première version, rejoué de bout en bout :
  1. contrôle des séries Bloomberg (Tankan, IPC, VIX) contre la table mensuelle, et des dates de publication ;
  2. réglage Optuna sur l'entraînement 1999-2021 (TimeSeriesSplit), comme le notebook 02 ;
  3. calibration de Platt et choix du modèle sur la validation 2022-2023, règle du notebook 03 ;
  4. une évaluation sur le test 2024-2026, au format du notebook 03b.

Le test 2024-2026 a déjà été ouvert par la première version : ce résultat est une exploration après coup, pas un
test à l'aveugle. La décision et l'évaluation figées de runs/ ne sont pas touchées : tout s'écrit dans runs/v2_bloomberg/.

Usage : NIKKEI_CONFIG=config_bloomberg.yaml python -m src.v2_bloomberg [--essais 50] [--sans-reglage]
"""
import argparse
import json
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss

from .config import CFG, COLONNES, CONFIG, ROOT, RUNS, commit_courant
from .data import charger_mensuel, charger_table
from .decision import MARGE, actions, appliquer_platt, apprendre_platt, couts_entrainement, rendements, seuils

MODELES = ["logit", "rf", "lgbm"]
NOMS = {"logit": "logistique", "rf": "Random Forest", "lgbm": "LightGBM", "moyenne": "moyenne des trois"}


# ---------- 1. les séries Bloomberg ----------
def _serie_bloomberg(fichier: str) -> pd.Series:
    x = pd.read_excel(ROOT / CFG["data"]["bloomberg"] / fichier, header=3)
    x.columns = ["date", "valeur"]
    return x.assign(date=pd.to_datetime(x["date"])).set_index("date")["valeur"].astype(float)


def controle_bloomberg() -> dict:
    """Compare les séries Bloomberg à la table mensuelle, en tenant compte de leur date de publication."""
    mensuel = charger_mensuel()
    mois = mensuel.index.to_period("M")
    rapport = {}

    # Tankan : Bloomberg date l'enquête à la fin du trimestre ; elle n'est connue qu'au mois suivant
    # (début avril, juillet, octobre ; mi-décembre pour l'enquête de décembre, utilisée en janvier par prudence).
    tankan = _serie_bloomberg("Trimestre_JNTSMFG_INDEX_HP.xlsx")
    publie = pd.Series(tankan.to_numpy(), index=tankan.index.to_period("M") + 1)
    table = pd.Series(mensuel["tankan_large_manuf"].to_numpy(), index=mois)
    communs = publie.index.intersection(table.index)
    egaux = int((publie[communs] == table[communs]).sum())
    naif = pd.Series(tankan.to_numpy(), index=tankan.index.to_period("M"))
    communs_naif = naif.index.intersection(table.index)
    rapport["tankan"] = {
        "trimestres_compares": int(len(communs)), "valeurs_identiques": egaux,
        "regle": "valeur de l'enquête du trimestre utilisée à partir du mois suivant sa date Bloomberg",
        "erreur_si_date_bloomberg_prise_telle_quelle": f"{int((naif[communs_naif] != table[communs_naif]).sum())} trimestres "
                                                       "auraient utilisé l'enquête avant sa publication"}

    # IPC : publié le mois suivant ; la table le recalcule à partir de l'indice, avec un mois de décalage
    ipc = _serie_bloomberg("Monthly_JNCPIYOY_INDEX_HP.xlsx")
    ipc_m = pd.Series(ipc.to_numpy(), index=ipc.index.to_period("M"))
    table_ipc = pd.Series(mensuel["cpi_all_yoy"].to_numpy(), index=mois).dropna()
    corr = {f"decalage_{k}": float(ipc_m.shift(k, freq="M").reindex(table_ipc.index).corr(table_ipc)) for k in range(3)}
    ecart = (ipc_m.shift(1, freq="M").reindex(table_ipc.index) - table_ipc).abs().dropna()
    rapport["ipc"] = {"mois_compares": int(len(ecart)), "correlations": corr,
                      "ecart_moyen_points": float(ecart.mean()), "ecart_max_points": float(ecart.max()),
                      "lecture": "même série que Bloomberg une fois décalée d'un mois (sa publication), recalculée à partir de l'indice ; "
                                 "l'IPC n'e

In [14]:
cb = lire_json("runs/v2_bloomberg/controle_bloomberg.json")
for serie, contenu in cb.items():
    print(serie, "→", {k: v for k, v in (contenu.items() if isinstance(contenu, dict) else [])
                       if not isinstance(v, (list, dict))})

tankan → {'trimestres_compares': 112, 'valeurs_identiques': 112, 'regle': "valeur de l'enquête du trimestre utilisée à partir du mois suivant sa date Bloomberg", 'erreur_si_date_bloomberg_prise_telle_quelle': "99 trimestres auraient utilisé l'enquête avant sa publication"}
ipc → {'mois_compares': 202, 'ecart_moyen_points': 0.054369069306930695, 'ecart_max_points': 0.41578899999999996, 'lecture': "même série que Bloomberg une fois décalée d'un mois (sa publication), recalculée à partir de l'indice ; l'IPC n'est pas une variable du modèle"}
vix → {'mois_compares': 334, 'ratio_median_bloomberg_sur_vix': 0.8563123565861241, 'ratio_min': 0.5993942636986301, 'ratio_max': 1.2381917554423343, 'decision': 'non utilisé : export « EUR, as exported », converti au taux de change ; le VIX en points (Cboe) de la table est gardé'}
calendrier → {'lignes': 5002, 'couvre': '1999-01-13 → 2006-12-27', 'limite': 'export tronqué à 5 000 lignes : à refaire par périodes pour couvrir 2007-2026'}


## 4 · Améliorations sans surapprentissage (V3)

**Ce que fait le fichier.** Il fige d'abord un protocole (variantes, réglages, périodes, critères) avec son empreinte
SHA-256, puis réentraîne chaque variante mois après mois sur 2010-2023. Une variante n'est retenue que si elle bat la
référence au test de Diebold-Mariano corrigé par Holm **et** sur les deux sous-périodes. Résultat : 0 levier retenu
sur 17.

**Concepts.** Walk-forward, Diebold-Mariano, Holm, bootstrap par blocs du Sharpe, protocole figé.

In [15]:
montrer("src/ameliorations.py")

**`src/ameliorations.py`** · 375 lignes

"""Troisième étape, après coup : appliquer les leviers d'amélioration sans surapprendre.

Le protocole (variantes, réglages, périodes, critères) est écrit dans runs/v3_ameliorations/protocole.json
AVANT tout calcul de résultat, avec son empreinte SHA-256. L'évaluation refuse de tourner si le protocole
du code ne correspond plus à celui qui a été figé.

Principes contre le surapprentissage :
  - chaque variante est réentraînée chaque mois sur tous les mois connus depuis 1999 (fenêtre croissante) ;
  - tous les réglages sont fixés à l'avance, aucun réglage automatique (pas d'Optuna) ;
  - la période de jugement est 2010-2023 ; une variante n'est retenue que si elle bat la référence
    après correction de Holm ET sur les deux sous-périodes 2010-2016 et 2017-2023 ;
  - 2024-2026 a déjà été ouvert par la V1 et la V2 : affiché à titre indicatif, jamais utilisé pour choisir.

Usage :
  python -m src.ameliorations --figer     écrit le protocole (refusé s'il existe déjà)
  python -m src.ameliorations             calcule tout, après avoir vérifié le protocole figé
"""
import argparse
import hashlib
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import yaml
from joblib import Parallel, delayed
from scipy.stats import norm
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .config import COLONNES, ROOT, SEED, commit_courant
from .decision import MARGE, actions, couts_entrainement
from .pipeline import construire_pipeline

V2 = yaml.safe_load(open(ROOT / "config_bloomberg.yaml"))
DOSSIER = ROOT / "runs" / "v3_ameliorations"
FICHIER_PROTOCOLE = DOSSIER / "protocole.json"

# Six variables choisies pour une raison économique connue avant les données, pas pour leur score :
ECONOMIQUES = ["n225_mom_12m",      # momentum 12 mois (Moskowitz, Ooi et Pedersen, 2012)
               "jgb_slope_10y2y",   # pente des taux japonais 10 ans - 2 ans (Fama et French, 1989)
               "us_slope_10y3m",    # pente des taux américains, indicateur avancé du cycle mondial
               "d_cli_jpn",         # variation de l'indicateur avancé OCDE du Japon
               "vix",               # appétit pour le risque mondial
               "n225_vol_ewma"]     # volatilité récente du Nikkei (arbitrage rendement-risque)
AJOUTEES = ["jgb_slope_10y2y", "us_slope_10y3m", "d_cli_jpn"]       # prises dans dataset_monthly2.csv

C_PROJET, C_FORT, ALPHA_RIDGE = 0.03, 0.003, 100.0
VOL_CIBLE, FRAIS, MELANGE = 0.15, 0.0005, 0.5
TIRAGES, BLOC, SEUIL = 5000, 12, 0.05

PERIODES = {"2010-2023 (jugement)": ("2010-01-01", "2023-12-31"),
            "2010-2016": ("2010-01-01", "2016-12-31"),
            "2017-2023": ("2017-01-01", "2023-12-31"),
            "2024-2026 (déjà vu, indicatif)": ("2024-01-01", "2026-09-30")}
JUGEMENT, SOUS_PERIODES, DEJA_VU = "2010-2023 (jugement)", ["2010-2016", "2017-2023"], "2024-2026 (déjà vu, indicatif)"

PROBABILITES = {
    "constante": "fréquence des baisses sur tout le passé connu (la référence)",
    "logistique du projet": f"13 variables du projet + régimes k-means, L2, C = {C_PROJET} (valeur retenue par la V1)",
    "logistique très pénalisée": f"mêmes variables, C = {C_FORT} : dix fois plus tirée vers la constante",
    "logistique du projet, mélangée à la constante": f"moyenne {MELANGE:.0%} logistique du projet, {1 - MELANGE:.0%} constante",
    "logistique économique": f"6 variables économiques, sans news ni régimes, L2, C = {C_PROJET}",
    "logistique économique, mélangée à la constante": f"moyenne {MELANGE:.0%} logistique économique, {1 - MELANGE:.0%} constante",
    "ridge sur le rendement": f"régression ridge (alpha = {ALPHA_RIDGE:g}) du rendement suivant sur les 6 variables économiques, "
                              "puis P(baisse) = Φ(-rendement prévu /

## 5 · Bancs d'essai (explorations après coup)

Ces scripts répondent à des questions posées **après** le test. Ils ne remplacent pas le résultat figé.
- `banc_essai.py` : beaucoup de modèles réentraînés mois après mois (constante, AR(1), lissage, tendance, logistique).
- `banc_essai_jour.py` : la même question sur 4 118 séances journalières, avec les décalages horaires Tokyo/New York.
- `banc_essai_long.py` : apprendre depuis 2000 ou depuis 1970 aide-t-il ? (prix seulement).
- `banc_essai_normalisation.py` : standardiser sur une fenêtre glissante (24, 36, 60 mois) après le changement de régime.
- `banc_essai_ouverture.py` : une décision à l'ouverture de Tokyo sur l'ETF 1321, un produit réellement achetable.

**Concepts.** Walk-forward, références naïves, calibration (ECE), extrapolation et standardisation glissante.

In [16]:
montrer("src/banc_essai.py")

**`src/banc_essai.py`** · 232 lignes

"""Exploration après coup : beaucoup de modèles, réentraînés mois après mois (walk-forward).

Ce script ne remplace pas le résultat du projet. Le test 2024-2026 a déjà été ouvert une fois
(runs/evaluation_test.json) : tout ce qui sort d'ici est une exploration faite en connaissant la suite.

Principe : pour prédire le mois t, chaque modèle n'apprend que sur les mois strictement antérieurs.
Deux fenêtres sont comparées : tout le passé (« croissante ») ou les 60 derniers mois (« glissante »).
Le choix du « meilleur » modèle se fait sur les mois d'avant le test (2016-2023) ; les 33 mois de test
servent ensuite à voir si ce choix tenait.

Usage : python -m src.banc_essai     écrit runs/exploration_apres_coup.json et ..._predictions.csv
"""
import json
import warnings
from datetime import datetime

import numpy as np
import optuna
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import norm, spearmanr
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss
from sklearn.model_selection import TimeSeriesSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from .config import CFG, COLONNES, RUNS, SEED, commit_courant
from .data import charger_table
from .decision import MARGE, actions, couts_entrainement, rendements
from .pipeline import construire_pipeline, proposer

DEPART = 72            # mois d'historique exigés avant la première prédiction (février 2016)
GLISSANTE = 60         # largeur de la fenêtre glissante, en mois
SORTIE = RUNS / "exploration_apres_coup.json"
PREDICTIONS = RUNS / "exploration_apres_coup_predictions.csv"
PRIX = CFG["features"]["price"] + ["n225_ret_1m"]
SANS_NEWS = [c for c in COLONNES if not c.startswith("news_")]


def officiel(nom: str):
    """Modèle du projet, avec les hyperparamètres retenus par Optuna (runs/trials_log.csv)."""
    journal = pd.read_csv(RUNS / "trials_log.csv")
    ligne = journal[(journal["cible"] == "nikkei") & (journal["modele"] == nom) & journal["retenu"]].iloc[0]
    return proposer(optuna.trial.FixedTrial(json.loads(ligne["params"])), nom)


def simple(modele, colonnes=COLONNES) -> tuple:
    """Imputation et standardisation sur un sous-ensemble de colonnes, sans les régimes k-means."""
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", modele)]), colonnes


# Chaque entrée : nom lisible -> fonction qui renvoie (pipeline, colonnes). Tout est réajusté à chaque mois.
APPRIS = {
    "logistique L2 (modèle du projet)": lambda: (construire_pipeline(officiel("logit")), COLONNES),
    "logistique L1": lambda: (construire_pipeline(LogisticRegression(
        C=0.1, penalty="l1", solver="saga", max_iter=5000, random_state=SEED)), COLONNES),
    "logistique très pénalisée": lambda: (construire_pipeline(LogisticRegression(C=0.003, max_iter=2000)), COLONNES),
    "logistique sans les news": lambda: simple(LogisticRegression(C=0.03, max_iter=2000), SANS_NEWS),
    "logistique, prix seuls": lambda: simple(LogisticRegression(C=0.03, max_iter=2000), PRIX),
    "analyse discriminante linéaire": lambda: (construire_pipeline(LinearDiscriminantAnalysis(shrinkage="auto", solver="lsqr")), COLONNES),
    "Bayes naïf gaussien": lambda: (construire_pipeline(GaussianNB()), COLONNES),
    "15 plus proches voisins": lambda: (construire_pipeline(KNeighborsClassifier(n_neighbors=15)), COLONNES),
    "SVM à noyau gaussien": lambda: (construire_pipeline(SVC(C=0.5, probability=True, random_state=SEED)), COLONNES),
    "Random Forest (projet)": lambda: (construire_pip

In [17]:
montrer("src/banc_essai_jour.py")

**`src/banc_essai_jour.py`** · 190 lignes

"""Exploration après coup, sur la base journalière (4 118 séances, 2010-2026).

Question : avec vingt fois plus de lignes que la table mensuelle, prédit-on mieux le sens du Nikkei ?
Ce script ne remplace pas le résultat du projet (runs/evaluation_test.json) ; le test a déjà été ouvert.

Précaution sur les horaires. Dans la base, la ligne du jour j contient la clôture de Wall Street du jour j,
connue plusieurs heures APRÈS la clôture de Tokyo. Tout ce qui ne vient pas de la Bourse de Tokyo est donc
décalé d'un jour quand la décision se prend à la clôture. Une expérience garde exprès Wall Street non décalé
pour montrer ce que ce piège rapporte, et une autre le rend honnête en décidant à l'ouverture du lendemain.

Usage : python -m src.banc_essai_jour     écrit runs/exploration_journaliere.json
"""
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .config import ROOT, RUNS, SEED, commit_courant

SORTIE = RUNS / "exploration_journaliere.json"
DEBUT = "2016-01-01"                 # première prédiction : six ans d'historique au moins
TEST = ("2024-01-01", "2026-09-30")  # mêmes mois que le test du projet
REFAIRE = 20                         # réentraînement toutes les 20 séances (un mois environ)
VOL_CIBLE = 0.15                     # volatilité annuelle visée par la stratégie de dosage


def charger() -> pd.DataFrame:
    for dossier in [ROOT / "data" / "2_construites"]:
        if (dossier / "dataset_daily_v2.csv").exists():
            return pd.read_csv(dossier / "dataset_daily_v2.csv", parse_dates=["date"]).set_index("date")
    raise SystemExit("dataset_daily_v2.csv introuvable (cherché dans data/2_construites/)")


def variables(d: pd.DataFrame) -> dict:
    """Trois familles de variables, selon le moment où elles sont connues."""
    tokyo = d[["n225_ret_1d", "n225_ret_5d", "n225_ret_20d", "n225_ret_60d", "n225_mom_12m", "n225_dist_sma20",
               "n225_dist_sma50", "n225_dist_sma200", "n225_vol_20d", "n225_vol_ewma", "n225_rsi14", "n225_macd_hist",
               "n225_bb_pctb", "n225_atr14_pct", "boj_days_to_next", "boj_days_since_last", "boj_meeting_day"]].copy()
    tokyo["taiex_r1"], tokyo["taiex_r5"] = d["taiex"].pct_change(), d["taiex"].pct_change(5)   # Taipei ferme avant Tokyo

    ailleurs = pd.DataFrame(index=d.index)
    for nom, jours in [("sp500", [1, 5, 20]), ("msci_em_eem", [5]), ("msci_japan_etf", [1, 5]), ("shanghai", [5]),
                       ("nifty50", [5]), ("eurjpy", [5, 20]), ("usdjpy", [5, 20])]:
        for j in jours:
            ailleurs[f"{nom}_r{j}"] = d[nom].pct_change(j)
    ailleurs["vix"], ailleurs["vix_d5"] = d["vix"], d["vix"].diff(5)
    for nom in ["us10y", "jgb10y", "spread_us_jgb10y"]:
        ailleurs[f"{nom}_d20"] = d[nom].diff(20)
    news = ["news_japan_sent_5d", "news_japan_sent_20d", "news_boj_sent_20d", "news_japan_neg_share_5d",
            "gdelt_japan_tone_5d", "gdelt_japan_attention"]
    ailleurs[news] = d[news]
    ailleurs["news_dispo"] = d["news_japan_sent_20d"].notna().astype(int)

    soir = pd.DataFrame({"sp500_soir": d["sp500"].pct_change(), "vix_soir": d["vix"].diff(),
                         "etf_japon_soir": d["msci_japan_etf"].pct_change(), "us10y_soir": d["us10y"].diff()})
    return {"tokyo": tokyo, "ailleurs_veille": ailleurs.shift(1), "wall_street_du_soir": soir}


def modeles() -> dict:
    prep = lambda m: Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", m)])
    return {"logistique L2": prep(LogisticRegression(C=0.01, max_iter=2000)),
            "LightGBM bridé": prep(LGBMClassifier(n_estimators=150, learning_rate=0.02, num_leaves=4, min_child_samples=100,
                   

In [18]:
montrer("src/banc_essai_long.py")

**`src/banc_essai_long.py`** · 115 lignes

"""Exploration après coup : apprendre sur plus de mois (depuis 2000, ou depuis 1970) aide-t-il ?

Seul le cours de clôture du Nikkei existe sur une aussi longue période dans nos fichiers
(data/1_brutes/nikkei225_long.csv, depuis 1970). Les variables sont donc tirées des prix seulement.
La comparaison porte sur la date de départ de l'apprentissage, toutes choses égales par ailleurs.
Ce script ne remplace pas le résultat du projet (runs/evaluation_test.json) ; le test a déjà été ouvert.

Usage : python -m src.banc_essai_long     écrit runs/exploration_historique_long.json
"""
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .config import ROOT, RUNS, SEED, commit_courant

SORTIE = RUNS / "exploration_historique_long.json"
DEPARTS = ["2010-02", "2000-01", "1985-01", "1971-06"]        # premier mois d'apprentissage
PERIODES = {"avant_test_2016_2023": ("2016-02", "2023-12"), "test_2024_2026": ("2024-01", "2026-09")}


def table_mensuelle() -> pd.DataFrame:
    """Variables de prix en fin de mois et sens du mois suivant, à partir des clôtures journalières."""
    for dossier in [ROOT / "data" / "1_brutes"]:
        if (dossier / "nikkei225_long.csv").exists():
            jour = pd.read_csv(dossier / "nikkei225_long.csv", parse_dates=[0], index_col=0)["close"].dropna()
            break
    else:
        raise SystemExit("nikkei225_long.csv introuvable")
    r_jour = jour.pct_change()
    mois = jour.groupby(jour.index.to_period("M")).last()
    r = mois.pct_change()
    t = pd.DataFrame({"ret_1m": r, "mom_3m": mois.pct_change(3), "mom_6m": mois.pct_change(6), "mom_12m": mois.pct_change(12),
                      "ecart_moyenne_10m": mois / mois.rolling(10).mean() - 1, "recul_depuis_le_plus_haut_12m": mois / mois.rolling(12).max() - 1,
                      "vol_1m": r_jour.groupby(r_jour.index.to_period("M")).std() * np.sqrt(252),
                      "vol_6m": r.rolling(6).std() * np.sqrt(12)})
    t["rendement_suivant"] = r.shift(-1)
    t["y_baisse"] = (t["rendement_suivant"] <= 0).astype(float).where(t["rendement_suivant"].notna())
    return t.dropna()


def modeles() -> dict:
    prep = lambda m: Pipeline([("scale", StandardScaler()), ("model", m)])
    return {"logistique L2": prep(LogisticRegression(C=0.03, max_iter=2000)),
            "logistique très pénalisée": prep(LogisticRegression(C=0.003, max_iter=2000)),
            "LightGBM bridé": prep(LGBMClassifier(n_estimators=100, learning_rate=0.02, num_leaves=4, min_child_samples=30,
                                                   subsample=0.8, subsample_freq=1, reg_lambda=5.0, random_state=SEED,
                                                   n_jobs=1, verbose=-1))}


def marcher(t: pd.DataFrame, depart: str) -> pd.DataFrame:
    """Réentraînement chaque mois sur tous les mois depuis `depart`, prédiction du mois suivant."""
    X, y = t.drop(columns=["rendement_suivant", "y_baisse"]), t["y_baisse"].astype(int)
    lignes = []
    for mois in t.loc[PERIODES["avant_test_2016_2023"][0]:PERIODES["test_2024_2026"][1]].index:
        passe = (t.index >= pd.Period(depart, "M")) & (t.index < mois)
        p = {"constante": y[passe].mean()}
        for nom, pipe in modeles().items():
            p[nom] = pipe.fit(X[passe], y[passe]).predict_proba(X.loc[[mois]])[0, 1]
        lignes.append({"mois": mois, "y": y[mois], "mois_d_apprentissage": int(passe.sum()), **p})
    return pd.DataFrame(lignes).set_index("mois")


def mesurer(pred: pd.DataFrame, reference: pd.DataFrame) -> dict:
    """Écart de log-loss avec la constante apprise depuis 2010 (la même référence pour tous les départs)."""
    perte = lambda y, p: -(y * np.log(p) + (1 - y) * np.log(1 - p))
    sortie = {}
    for cle, (a, b) in PERIODES.items

In [19]:
montrer("src/banc_essai_normalisation.py")

**`src/banc_essai_normalisation.py`** · 69 lignes

"""Exploration après coup : normaliser les variables sur une fenêtre glissante aide-t-il après le changement de régime ?

Dans le projet, chaque variable est centrée et réduite avec la moyenne et l'écart-type de tout l'entraînement.
Après 2022, plusieurs variables sortent de cette plage (le portage est à six écarts-types). Ici, chaque valeur est
comparée aux 24, 36 ou 60 derniers mois seulement : z = (x - moyenne récente) / écart-type récent, sans regarder le futur.
Les modèles sont ensuite réentraînés chaque mois, comme dans src.banc_essai. Le test avait déjà été ouvert.

Usage : python -m src.banc_essai_normalisation     écrit runs/exploration_normalisation.json
"""
import json
import warnings
from datetime import datetime

import pandas as pd
from joblib import Parallel, delayed

from .banc_essai import DEPART, marcher, perte, resumer
from .config import COLONNES, ROOT, RUNS, commit_courant
from .data import charger_table

SORTIE = RUNS / "exploration_normalisation.json"
FENETRES = [24, 36, 60]
MODELES = ["logistique L2 (modèle du projet)", "logistique très pénalisée", "logistique L1", "Random Forest (projet)", "LightGBM (projet)"]


def normaliser(table: pd.DataFrame, mois: int) -> pd.DataFrame:
    """Chaque variable continue est exprimée en écarts-types par rapport à ses `mois` derniers mois (mois courant inclus)."""
    t = table.copy()
    for col in [c for c in COLONNES if t[c].nunique() > 2]:           # l'indicateur de présence des news reste tel quel
        glissant = t[col].rolling(mois, min_periods=12)
        t[col] = ((t[col] - glissant.mean()) / glissant.std()).clip(-5, 5)
    return t


def main() -> None:
    warnings.filterwarnings("ignore")
    table = charger_table()
    table = table[table["bloc"] != "à prédire"]
    variantes = {"tout l'entraînement (projet)": table} | {f"{m} derniers mois": normaliser(table, m) for m in FENETRES}
    plan = [(v, nom) for v in variantes for nom in MODELES]
    morceaux = Parallel(n_jobs=-1)(delayed(marcher)(nom, "croissante", variantes[v]) for v, nom in plan)
    const = marcher("constante (fréquence passée)", "croissante", table).set_index("date")
    periodes = {"avant_test_2016_2023": table.index[DEPART:][(table["bloc"].iloc[DEPART:] != "test").to_numpy()],
                "test_2024_2026": table.index[table["bloc"] == "test"]}
    resultats = []
    for (v, nom), pred in zip(plan, morceaux):
        pred, ligne = pred.set_index("date"), {"normalisation": v, "modele": nom}
        for cle, mois in periodes.items():
            bloc = table.loc[mois]
            ref = pd.Series(perte(bloc["y_baisse"], const.loc[mois, "p_baisse"]), index=mois)
            ligne[cle] = resumer(pred.loc[mois], bloc, ref)
        resultats.append(ligne)
    sortie = {"avertissement": "Exploration après coup : le test 2024-2026 avait déjà été ouvert. Ces chiffres ne remplacent pas "
                               "runs/evaluation_test.json, qui reste le résultat du projet.",
              "date": datetime.now().strftime("%Y-%m-%d %H:%M"), "commit_du_code": commit_courant(),
              "methode": {"normalisation": "z = (x - moyenne des N derniers mois) / écart-type des N derniers mois, borné à ±5",
                          "fenetres_mois": FENETRES, "reentrainement": "chaque mois, sur tout le passé",
                          "reference": "constante (fréquence passée)"},
              "resultats": resultats}
    SORTIE.write_text(json.dumps(sortie, indent=2, ensure_ascii=False))
    for r in sorted(resultats, key=lambda r: r["test_2024_2026"]["log_loss"]):
        a, t = r["avant_test_2016_2023"], r["test_2024_2026"]
        print(f"{r['normalisation']:30s} {r['modele'][:34]:34s} avant {a['ecart_vs_constante']:+.3f} ± {a['erreur_type_de_l_ecart']:.3f} | "
              f"test LL {t['log_loss']:.3f} écart {t['ecart_vs_constante']:+.3f} ± {t['erreur_type_de_l_ecart']:.3f} PR-AUC {t['pr_auc']:.2f} règle {t['cumul_regle']:+.1%}")
    print(f"écrit : {SORTIE.relative_to(ROOT)}")


if __name

In [20]:
montrer("src/banc_essai_ouverture.py")

**`src/banc_essai_ouverture.py`** · 115 lignes

"""Exploration après coup : une décision prise chaque matin à l'ouverture de Tokyo, sur un produit réellement achetable.

Sur l'indice, un signal apparaissait à l'ouverture (src.banc_essai_jour), mais l'indice ouvre avec des cours en partie
périmés. Ici, on le teste sur l'ETF Nikkei 225 coté à Tokyo (code 1321), dont l'ouverture est un vrai prix d'échange
(data/1_brutes/nikkei_etf_1321.csv, Yahoo Finance).

Décision : à l'ouverture du jour J+1, en connaissant tout ce qui était connu à la clôture de Tokyo du jour J,
la séance de Wall Street de la nuit, et le saut d'ouverture de l'ETF. Cible : l'ETF baisse-t-il entre l'ouverture et la
clôture de J+1 ? Le test 2024-2026 avait déjà été ouvert : ces chiffres ne remplacent pas le résultat du projet.

Usage : python -m src.banc_essai_ouverture     écrit runs/exploration_ouverture.json
"""
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from .banc_essai_jour import TEST, charger, marcher, mesurer, variables
from .config import ROOT, RUNS, commit_courant

SORTIE = RUNS / "exploration_ouverture.json"
FRAIS = [0.0, 0.0002, 0.0005, 0.0010]          # coût d'un aller-retour (achat à l'ouverture, revente à la clôture)
MARGE = 0.03                                   # on ne prend position que si la probabilité s'écarte de 50 % d'au moins 3 points


def charger_etf() -> pd.DataFrame:
    for dossier in [ROOT / "data" / "1_brutes"]:
        if (dossier / "nikkei_etf_1321.csv").exists():
            return pd.read_csv(dossier / "nikkei_etf_1321.csv", parse_dates=[0], index_col=0)
    raise SystemExit("nikkei_etf_1321.csv introuvable (cherché dans data/1_brutes/)")


def bilan(gains: pd.Series, positions: pd.Series) -> dict:
    """Résultat d'une suite de paris d'une séance : `gains` nets, un par séance (0 quand on reste à l'écart)."""
    pris = positions != 0
    valeur = (1 + gains).cumprod()
    return {"seances_jouees": int(pris.sum()), "part_des_seances_%": float(pris.mean() * 100),
            "paris_gagnants_%": float((gains[pris] > 0).mean() * 100) if pris.any() else None,
            "gain_moyen_par_pari_pb": float(gains[pris].mean() * 1e4) if pris.any() else None,
            "cumul": float(valeur.iloc[-1] - 1), "perte_maximale": float((valeur / valeur.cummax() - 1).min()),
            "rendement_par_unite_de_risque": float(gains.mean() / gains.std() * np.sqrt(252)) if gains.std() > 0 else 0.0}


def main() -> None:
    warnings.filterwarnings("ignore")
    d, etf = charger(), charger_etf()
    etf = etf.reindex(d.index)
    ouverture, cloture = etf["Open"], etf["Close"]
    saut = (ouverture.shift(-1) / cloture - 1).where(lambda s: s.abs() < 0.15)       # connu à l'ouverture de J+1
    seance = cloture.shift(-1) / ouverture.shift(-1) - 1                             # ce qu'on cherche à prévoir
    seance = seance.where(etf["Volume"].shift(-1) > 0)
    v = variables(d)
    X = pd.concat([v["tokyo"], v["ailleurs_veille"], v["wall_street_du_soir"]], axis=1).assign(saut_d_ouverture=saut)
    y = (seance <= 0).astype(float).where(seance.notna() & saut.notna())

    pred = marcher(X, y, 1)
    mesures = mesurer(pred, 1)

    # Stratégies d'une séance, sur le test : +1 = acheter à l'ouverture, -1 = vendre à découvert, 0 = rester à l'écart
    test = pred.loc[TEST[0]:TEST[1]]
    r = seance.reindex(test.index)
    p_hausse = 1 - test["moitié logistique, moitié LightGBM"]
    nuit = d["sp500"].pct_change().reindex(test.index)
    regles = {"modèle (moitié logistique, moitié LightGBM)": np.sign(p_hausse - 0.5).where((p_hausse - 0.5).abs() >= MARGE, 0.0),
              "modèle, achats seulement": (p_hausse >= 0.5 + MARGE).astype(float),
              "règle simple : suivre Wall Street de la nuit": np.sign(nuit).fillna(0.0),
              "règle simple : suivre le saut d'ouverture": np.sign(saut.reindex(test.index)).fillna(0.0),
              "toujours acheter à l'ouverture, revendre à la clôture": pd.Series(1.0, index=test.index)}
    strategies

## 5 · Prévoir l'agitation plutôt que le sens

**Ce que font les fichiers.** Le sens du marché ne se prévoit pas, mais sa volatilité oui : elle vient par périodes.
`volatilite.py` prévoit la volatilité du mois suivant (GARCH, régression sur la volatilité passée) et la probabilité
d'un mois agité. `signal_risque.py` en déduit chaque jour la part à investir (100 %, 50 %, 0 %) ;
`signal_risque_variantes.py` teste trois améliorations écrites avant de voir leurs résultats.

**Concepts.** Volatilité conditionnelle, GARCH, R² hors échantillon, ciblage de volatilité, ratio de Sharpe.

In [21]:
montrer("src/volatilite.py")

**`src/volatilite.py`** · 230 lignes

"""Second volet, ajouté après coup : prévoir l'agitation du Nikkei plutôt que son sens.

Le sens du mois suivant ne se prévoit pas (runs/evaluation_test.json, résultat figé du projet).
Ce script pose une autre question sur les mêmes mois et le même découpage : quelle sera la volatilité
du mois prochain, et ce mois sera-t-il agité ? Il a été écrit après l'ouverture du test.

Trois parties :
  1. prévision de la volatilité du mois suivant, sept méthodes réentraînées chaque mois ;
  2. classification « mois agité » (volatilité réalisée au-dessus de 20 %), avec la démarche du projet ;
  3. usage : le risque annoncé est-il fiable, et que donne une position dosée selon ce risque ?

Usage : python -m src.volatilite     écrit runs/exploration_volatilite.json et ..._predictions.csv
"""
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from scipy.optimize import minimize
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .banc_essai import DEPART, ece, perte
from .banc_essai_jour import charger as charger_jours
from .config import ROOT, RUNS, SEED, commit_courant
from .data import charger_table

SORTIE = RUNS / "exploration_volatilite.json"
PREDICTIONS = RUNS / "exploration_volatilite_predictions.csv"
SEUIL_AGITE = 0.20        # un mois est « agité » si sa volatilité réalisée dépasse 20 % par an (la moyenne du Nikkei)
VOL_CIBLE = 0.15          # volatilité visée par la position dosée
RETENU = "régression : 1, 3, 6 mois + VIX + rendement"       # méthode utilisée dans la partie « usage »
HAR = ["lv_1m", "lv_3m", "lv_6m"]
TOUTES = HAR + ["lv_ewma", "lv_vix", "n225_ret_1m", "boj_meeting_next_month"]


def construire() -> tuple[pd.DataFrame, pd.Series]:
    """Table mensuelle du projet, enrichie des volatilités réalisées calculées sur les séances."""
    jours = charger_jours()
    r = jours["n225_close"].pct_change().dropna()
    mois = r.index.to_period("M")
    annuel = np.sqrt(252)
    fin = pd.DataFrame({"lv_1m": np.log(r.groupby(mois).std() * annuel),
                        "lv_3m": np.log((r.rolling(63).std() * annuel).groupby(mois).last()),
                        "lv_6m": np.log((r.rolling(126).std() * annuel).groupby(mois).last())})
    complet = r.groupby(mois).count() >= 15                       # un mois entamé n'a pas de volatilité réalisée
    fin["lv_suivant"] = fin["lv_1m"].where(complet).shift(-1)
    t = charger_table()
    t = pd.concat([t, fin.reindex(t.index.to_period("M")).set_index(t.index)], axis=1)
    t["lv_ewma"], t["lv_vix"] = np.log(t["n225_vol_ewma"]), np.log(t["vix"] / 100)
    t["agite"] = (np.exp(t["lv_suivant"]) > SEUIL_AGITE).astype(float).where(t["lv_suivant"].notna())
    return t, r


class Garch:
    """GARCH(1,1) gaussien, variance de long terme fixée à la variance observée, ajusté par maximum de vraisemblance."""

    def __init__(self):
        self.alpha, self.beta = 0.08, 0.90

    @staticmethod
    def _variances(r, alpha, beta):
        longue = r.var()
        s2 = np.empty(len(r) + 1)
        s2[0] = longue
        for i, x in enumerate(r):
            s2[i + 1] = longue * (1 - alpha - beta) + alpha * x * x + beta * s2[i]
        return s2, longue

    def ajuster(self, r: np.ndarray) -> "Garch":
        def moins_vraisemblance(theta):
            s2, _ = self._variances(r, *theta)
            return 0.5 * np.sum(np.log(s2[:-1]) + r * r / s2[:-1])
        res = minimize(moins_vraisemblance, [self.alpha, self.beta], method="L-BFGS-B",
                       bounds=[(0.01, 0.3), (0.5, 0.98)], options={"maxiter": 60})
        if res.x.sum() < 0.999:
            self.alpha, self.beta = res.x
        return self

    def prevoir(self, r: np.ndarray, seances: int = 21) -> float:
        """Volatilité annual

In [22]:
montrer("src/signal_risque.py")

**`src/signal_risque.py`** · 295 lignes

"""Signal d'achat fondé sur le risque : chaque jour, on prévoit la volatilité du Nikkei sur le mois qui vient
et on en déduit combien investir (100 %, 50 % ou 0 %). On ne cherche plus à deviner la hausse ou la baisse.

En clair
- Le risque d'un marché (sa volatilité) varie par périodes : calme, puis agité, puis calme. Il se prévoit, le sens
  du marché non (voir src/banc_essai*.py).
- Chaque soir, après la clôture de Tokyo, on prévoit la volatilité des 20 prochaines séances.
- Risque prévu bas : acheter plein. Moyen : acheter à moitié. Très élevé : ne pas acheter (l'argent reste en euros).
- Pour ne pas acheter et vendre sans arrêt, un changement de niveau doit être confirmé trois soirs de suite.
- On exécute le lendemain à la clôture : aucun prix inconnu au moment de décider n'est utilisé.

Protocole, en deux temps (pour ne pas régler la règle sur les années qui servent à la juger)
  python -m src.signal_risque --figer     choisit le modèle et les seuils avec les seules données 2000-2015,
                                          écrit runs/signal_risque_regle.json (refuse d'écraser une règle existante)
  python -m src.signal_risque             applique la règle figée à 2003-2026 et écrit runs/signal_risque.json

Honnêteté : les années 2016-2026 n'ont pas servi aux choix, mais l'auteur les connaît, et une version mensuelle de
cette idée avait déjà été regardée sur 2024-2026 (src/volatilite.py). Ce volet s'ajoute au résultat officiel du
projet (runs/evaluation_test.json), il ne le remplace pas. Indices de prix sans dividendes (des deux côtés).

Données (data/1_brutes/depuis_2000) : Nikkei 225, S&P 500, VIX, dollar-yen (Yahoo Finance), euro-yen et Euribor
3 mois (BCE), taux 3 mois japonais (OCDE, à partir d'avril 2002 ; avant, la première valeur connue est reprise).
"""
import argparse
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

from .config import ROOT, RUNS, commit_courant
from .volatilite import Garch

DOSSIER = ROOT / "data" / "1_brutes" / "depuis_2000"
REGLE = RUNS / "signal_risque_regle.json"
SORTIE = RUNS / "signal_risque.json"
SERIE = RUNS / "signal_risque_journalier.csv"

DEBUT, FIN_REGLAGE = "2000-01-01", "2015-12-31"
PERIODES = {"2003-2015 (réglage)": ("2003-01-01", "2015-12-31"), "2016-2023": ("2016-01-01", "2023-12-31"),
            "2024-2026": ("2024-01-01", "2026-12-31"), "2003-2026 (tout)": ("2003-01-01", "2026-12-31")}
HORIZON = 20            # séances : la volatilité prévue est celle du mois qui vient
DECALAGE = 1            # on décide le soir t, on exécute à la clôture de t+1 : la position gagne le rendement de t+2
PAS = 20                # réentraînement toutes les 20 séances, sur tout le passé disponible
MINIMUM = 756           # trois ans d'historique avant la première prévision
QUANTILES = (0.60, 0.90)   # 60 % des jours « calmes », 30 % « moyens », 10 % « très risqués » sur 2003-2015
CONFIRMATION = 3        # soirs de suite dans la nouvelle zone avant de changer de niveau
FRAIS = 0.0005          # 0,05 % par 100 % acheté ou vendu (ETF Nikkei coté à Tokyo, écart compris)


# ---------------------------------------------------------------- données
def charger() -> pd.DataFrame:
    n = pd.read_csv(DOSSIER / "nikkei225_ohlc.csv", index_col=0, parse_dates=True).loc[DEBUT:].dropna(subset=["Close"])
    n = n[n["Close"].pct_change().abs().fillna(0) < 0.2]
    m = pd.read_csv(DOSSIER / "yahoo_marches.csv", index_col=0, parse_dates=True)
    ecb = pd.read_csv(DOSSIER / "ecb_eurjpy.csv", usecols=["TIME_PERIOD", "OBS_VALUE"], parse_dates=["TIME_PERIOD"]).set_index("TIME_PERIOD")["OBS_VALUE"].sort_index()
    eur = pd.read_csv(DOSSIER / "ecb_euribor3m.csv", usecols=["TIME_PERIOD", "OBS_VALUE"]).assign(d=lambda x: pd.to_datetime(x.TIME_PERIOD) + pd.offsets.MonthEnd(0)).set_index("d")["OBS_VALUE"].sort_index()
    jp = pd.read_csv(DOSSIER / "oecd_taux3m_japon.csv", usecols=["TIME_PERIOD", "OBS_VALUE"]).assign(d=

In [23]:
montrer("src/signal_risque_variantes.py")

**`src/signal_risque_variantes.py`** · 138 lignes

"""Trois améliorations du signal fondé sur le risque, plus leur combinaison, testées sans tricher.

Les variantes ont été écrites AVANT de voir leurs résultats, pour une raison de principe :
  A. Levier en période calme : part investie = c / variance prévue, jusqu'à 150 % (contrats à terme). c est choisi pour
     que la part investie moyenne vaille 100 % sur 2003-2015 (aucun rendement n'intervient dans ce choix).
  B. Sortir lentement, revenir vite : on remonte d'un niveau dès le premier soir, on descend après trois soirs ; et l'on
     ne sort entièrement que si le cours est aussi sous sa moyenne des 200 dernières séances (filtre de tendance).
  C. Décider seulement en fin de mois, avec la règle figée de src.signal_risque.
  D. Les trois ensemble : part c / variance prévue revue en fin de mois, jusqu'à 150 %, ramenée à 50 % au plus quand le
     cours est sous sa moyenne 200 séances et le risque très élevé.

Protocole : `--figer` calcule c avec les seules données 2000-2015 et écrit runs/signal_risque_variantes_regle.json ;
sans option, le script applique ces réglages figés à 2003-2026. Toutes les variantes sont publiées. Quatre essais au lieu
d'un augmentent la chance qu'un d'eux réussisse par hasard : un écart doit dépasser nettement son erreur type.
Volet après coup : il ne remplace pas le résultat du projet.
"""
import argparse
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from . import signal_risque as S
from .config import ROOT, RUNS, commit_courant

REGLE = RUNS / "signal_risque_variantes_regle.json"
SORTIE = RUNS / "signal_risque_variantes.json"
PLAFOND = 1.5
MOYENNE_LONGUE = 200


def fin_de_mois(w: pd.Series) -> pd.Series:
    """Garde la valeur du dernier jour de chaque mois, tenue tout le mois suivant."""
    dernier = w.index.to_series().groupby(w.index.to_period("M")).transform("max") == w.index.to_series()
    return w.where(dernier).ffill().bfill()


def variante_b(vol: pd.Series, tendance_baissiere: pd.Series, seuils) -> pd.Series:
    zone = pd.Series(np.select([vol < seuils[0], vol < seuils[1]], [1.0, 0.5], 0.0), index=vol.index)
    zone = zone.where(~((zone == 0.0) & ~tendance_baissiere.reindex(vol.index).fillna(False)), 0.5)   # pas de sortie totale en tendance haussière
    sortie, actuel, attente = [], 1.0, 0
    for z in zone.to_numpy():
        if z > actuel:
            actuel, attente = z, 0                       # revenir : tout de suite
        elif z < actuel:
            attente += 1
            if attente >= S.CONFIRMATION:                # sortir : après trois soirs
                actuel, attente = z, 0
        else:
            attente = 0
        sortie.append(actuel)
    return pd.Series(sortie, index=vol.index)


def positions(vol: pd.Series, d: pd.DataFrame, seuils, c: float) -> dict:
    baissier = d["close"] < d["close"].rolling(MOYENNE_LONGUE).mean()
    levier = (c / vol ** 2).clip(upper=PLAFOND)
    combine = fin_de_mois(levier)
    tres_risque = fin_de_mois(pd.Series((vol >= seuils[1]).astype(float), index=vol.index)) > 0
    combine = combine.where(~(tres_risque & fin_de_mois(baissier.reindex(vol.index).astype(float)).astype(bool)), combine.clip(upper=0.5))
    return {"acheter et garder": pd.Series(1.0, index=vol.index),
            "règle figée (jour, 3 niveaux)": S.niveaux(vol, seuils, S.CONFIRMATION),
            "A. levier en période calme (c / variance, jusqu'à 150 %)": levier,
            "B. sortir lentement, revenir vite, filtre de tendance": variante_b(vol, baissier, seuils),
            "C. décision en fin de mois": S.niveaux(vol, seuils, 1, fin_de_mois=True),
            "D. les trois ensemble": combine}


def prevoir(fin=None):
    d = S.charger()
    if fin:
        d = d.loc[:fin]
    regle = json.loads(S.REGLE.read_text())
    x, y = S.variables(d), S.cible(d)
    pred = S.marcher(d, x, y, fin)
    return d, regle, np.exp(pred[regle["modele"]])


def figer() -> None:
    if REGLE.exists():
        raise SystemEx

## 6 · Annexe news × prix

Étude préalable (dossier `notebooks/06_annexe_news_prix/`) : le signal des news et celui des prix, le temps qu'ils
mettent à se réaliser, des backtests de stratégies et un modèle de Markov caché pour les régimes.

**Concepts.** Corrélations décalées, backtest, HMM (états cachés, matrice de transition, algorithme de Viterbi).

In [24]:
montrer("notebooks/06_annexe_news_prix/nikkei_lab.py")

**`notebooks/06_annexe_news_prix/nikkei_lab.py`** · 1248 lignes

"""
nikkei_lab — moteur du notebook « Nikkei : signal news × signal prix ».

Contenu
-------
1. Chargement des données (marché + articles)                     -> load_data
2. Signal NEWS  (sentiment FinBERT des articles / ton GDELT)        -> news_signal
3. Signal PRIX  (tendance, momentum, moyennes mobiles, drawdown,   -> price_signal
                 régimes de marché par HMM)                        -> hmm_regimes (moteur : nikkei_hmm.py)
4. Combinaison + passage signal -> position + backtest réaliste     -> run_strategy
5. Analyse des horizons : en combien de temps un signal se réalise  -> horizon_profile, placebo_ic,
                                                                       decay_matrix, event_study, holding_curve
6. Optimisation : grille, IS/OOS, walk-forward, plateau, Deflated Sharpe
7. Méta-modèle ML walk-forward purgé (bonus)

Convention temporelle (importante pour éviter le biais d'anticipation)
--------------------------------------------------------------------
* Toute série indexée en t n'utilise que l'information disponible à la clôture de la séance t.
  Les articles sont déjà rattachés à leur séance (`session`) : ceux vus avant ~15h JST le jour t.
* La position décidée en t est exécutée à la clôture de t + exec_lag (exec_lag=1 par défaut),
  puis encaisse les rendements des jours suivants.
"""
from __future__ import annotations

import itertools
import math
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import stats
from scipy.ndimage import uniform_filter
from scipy.signal import lfilter

import nikkei_hmm as nh

TD = 252  # jours de bourse par an

# =============================================================================
# 0. Paramètres par défaut (tous modifiables depuis le notebook)
# =============================================================================
DEFAULT_PARAMS: dict = dict(
    # ---------------- NEWS ----------------
    news_source="articles",   # 'articles' (FinBERT) | 'gdelt' (ton GDELT) | 'blend'
    news_score="prob",        # score article : 'prob' (p_pos-p_neg) | 'vote' (+1/0/-1) | 'neg' | 'pos'
    news_mode="level",        # 'level' (niveau du sentiment) | 'momentum' (rapide - lent) | 'attention' (volume anormal)
    news_halflife=20,         # demi-vie (en séances) de la mémoire des news
    news_slow_mult=4,         # momentum/attention : demi-vie lente = halflife * slow_mult
    news_prior=5.0,           # shrinkage : nb d'articles « fictifs » à la moyenne long terme
    news_w_boj=1.0,           # poids des articles thème BoJ
    news_w_japan=1.0,         # poids des articles thème Japon
    news_min_conf=0.0,        # filtre : garde les articles avec 1 - p_neu >= min_conf
    news_conf_power=0.0,      # pondère chaque article par (1 - p_neu) ** power
    news_dedup_days=3,        # supprime un titre déjà vu dans les N jours (0 = off)
    # --- qualité de la collecte (voir collection_quality) ---
    news_gap_min_run=2,       # N séances d'affilée sans article (ou panne GDELT signalée) = trou de collecte
                              # -> donnée manquante (0 = off : une séance vide compte comme « calme »)
    news_burst_mult=3.0,      # pic = nb d'articles > mult × médiane des 60 séances précédentes, écrêté à ce niveau (0 = off)
    news_burst_confirm=True,  # un pic que l'attention GDELT (mesurée sur tout le corpus) ne confirme pas = donnée manquante
    news_burst_confirm_z=0.0, # seuil de confirmation : max(attention GDELT Japon, BoJ) en z > seuil
    news_norm="rolling",      # normalisation : 'rolling' | 'expanding' | 'none'
    news_z_window=250,        # fenêtre de normalisation (séances)
    news_gdelt_weight=0.5,    # poids du GDELT si news_source='blend'
    news_direction=1,         # +1 = suivre le sentiment, -1 = contrarien
    # ---------------- PRIX ----------------
    price_kind="tsmom",       # 'tsmom' | 'composite' | 'ma' | 'sma_dist' | 'breakout' | 'drawdown' | 'hmm'
    price_lookback=120,   

In [25]:
montrer("notebooks/06_annexe_news_prix/nikkei_hmm.py")

**`notebooks/06_annexe_news_prix/nikkei_hmm.py`** · 336 lignes

"""
nikkei_hmm — régimes du Nikkei par modèle de Markov caché (HMM) gaussien, sans biais d'anticipation.

Un HMM suppose que le marché passe d'un régime caché à l'autre (calme, agité…) selon une chaîne de Markov,
et que, dans chaque régime, les observations (rendement quotidien, volatilité réalisée…) suivent une loi
gaussienne qui lui est propre.

* `GaussianHMM`      : estimation par EM (Baum-Welch). Les récurrences avant / arrière sont calculées par
                       produits matriciels préfixes (log2(T) étapes vectorisées au lieu d'une boucle sur T).
* `hmm_walk_forward` : ré-estime le modèle toutes les `refit` séances sur les seules données passées et ne
                       garde que les probabilités FILTRÉES P(régime_t | données jusqu'à t). Les probabilités
                       lissées (forward-backward) et le chemin de Viterbi utilisent le futur : c'est le piège
                       classique des backtests HMM.
* `implied_timing`   : durée moyenne de chaque régime et vitesse à laquelle le modèle « oublie » le régime
                       courant (T50 / T90 implicites), à comparer aux temps de réalisation mesurés.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

LOG2PI = float(np.log(2 * np.pi))


# =============================================================================
# Récurrences avant / arrière vectorisées
# =============================================================================
def _prefix_products(M: np.ndarray):
    """P_t = M_0 @ M_1 @ … @ M_t pour tout t (balayage de Hillis-Steele, log2(T) produits vectorisés).
    Chaque P_t est divisé par son maximum ; la vraie valeur vaut P_t * exp(log_scale_t)."""
    P = np.array(M, dtype=float)
    m = np.maximum(P.max(axis=(1, 2)), 1e-300)
    P /= m[:, None, None]
    ls = np.log(m)
    d, n = 1, len(P)
    while d < n:
        Q = P[:-d] @ P[d:]
        m = np.maximum(Q.reshape(n - d, -1).max(axis=1), 1e-300)
        Q /= m[:, None, None]
        P[d:] = Q
        ls[d:] = ls[:-d] + ls[d:] + np.log(m)
        d *= 2
    return P, ls


def _emission_logpdf(X: np.ndarray, means: np.ndarray, variances: np.ndarray) -> np.ndarray:
    """log N(x_t | mu_k, diag(var_k)) -> (T, K). Une ligne avec une valeur manquante n'apporte aucune information."""
    bad = np.isnan(X).any(axis=1)
    Xf = np.where(np.isnan(X), 0.0, X)
    ll = -0.5 * (LOG2PI * X.shape[1] + np.log(variances).sum(1)[None, :]
                 + (((Xf[:, None, :] - means[None]) ** 2) / variances[None]).sum(2))
    ll[bad] = 0.0
    return ll


def _forward(logb: np.ndarray, startprob: np.ndarray, transmat: np.ndarray):
    """Probabilités filtrées P(s_t | x_0..t) et log-vraisemblance. Retourne aussi b et M pour l'arrière."""
    c = logb.max(axis=1)
    b = np.exp(logb - c[:, None])
    a0 = startprob * b[0]
    s0 = a0.sum()
    a0 = a0 / s0
    if len(b) == 1:
        return a0[None], float(np.log(s0) + c.sum()), b, None
    M = transmat[None, :, :] * b[1:, None, :]          # M_t = A diag(b_t), t = 1..T-1
    P, ls = _prefix_products(M)
    au = np.einsum("k,tkj->tj", a0, P)                  # ∝ alpha_t
    s = au.sum(1)
    alpha = np.vstack([a0, au / s[:, None]])
    loglik = float(np.log(s0) + np.log(s[-1]) + ls[-1] + c.sum())
    return alpha, loglik, b, M


def _posteriors(logb: np.ndarray, startprob: np.ndarray, transmat: np.ndarray):
    """E-step : probabilités lissées gamma_t, transitions attendues sum_t xi_t, log-vraisemblance."""
    alpha, loglik, b, M = _forward(logb, startprob, transmat)
    K = len(startprob)
    if M is None:
        return alpha, np.zeros((K, K)), loglik
    Q, _ = _prefix_products(np.transpose(M[::-1], (0, 2, 1)))   # Q_k = (M_{T-1-k} … M_{T-1})^T
    beta = Q.sum(axis=1)[::-1]                                   # beta_0 … beta_{T-2}
    beta = np.vstack([beta / beta.sum(1, keepdims=True), np.ones((1, K)) / K])
    g = alpha * beta
    g /= g.sum(1, keepdims=True)
    xi = alpha[:-1, :, None] * transmat[None] * (b[1:] * 

In [26]:
montrer("notebooks/06_annexe_news_prix/nikkei_plots.py")

**`notebooks/06_annexe_news_prix/nikkei_plots.py`** · 650 lignes

"""
nikkei_plots — graphiques et tableau de bord interactif du notebook.

Palette : instance de référence validée (clair ET sombre) — rôles fixes dans tout le notebook :
  stratégie = bleu · news = orange · prix = aqua · ML = jaune · buy & hold = gris.
Jamais de double axe : les séries d'échelles différentes vont dans des panneaux séparés.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.dates  # noqa: F401  (mpl.dates utilisé pour placer les étiquettes de fin)
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm, to_rgb
from matplotlib.ticker import FuncFormatter, PercentFormatter, FixedLocator

import nikkei_lab as nl

LIGHT = dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
             strategy="#2a78d6", news="#eb6834", price="#1baf7a", ml="#eda100", bench="#898781", bench2="#c3c2b7",
             div=["#a32b2b", "#e34948", "#f4b4b3", "#f0efec", "#b7d3f6", "#3987e5", "#184f95"],
             seq=["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DARK = dict(surface="#1a1a19", ink="#f0efec", ink2="#c3c2b7", muted="#898781", grid="#2c2c2a", axis="#383835",
            strategy="#3987e5", news="#d95926", price="#199e70", ml="#c98500", bench="#898781", bench2="#5a5955",
            div=["#f2a7a6", "#e66767", "#7a2f2f", "#383835", "#1c4a80", "#3987e5", "#b7d3f6"],
            seq=["#13325a", "#184f95", "#256abf", "#3987e5", "#6da7ec", "#9ec5f4", "#cde2fb"])
T: dict = dict(LIGHT)


def set_theme(dark: bool = False):
    """Applique le thème (clair par défaut ; dark=True pour un notebook en thème sombre)."""
    T.clear()
    T.update(DARK if dark else LIGHT)
    mpl.rcParams.update({
        "figure.facecolor": T["surface"], "axes.facecolor": T["surface"], "savefig.facecolor": T["surface"],
        "axes.edgecolor": T["axis"], "axes.labelcolor": T["ink2"], "axes.titlecolor": T["ink"],
        "axes.grid": True, "grid.color": T["grid"], "grid.linewidth": 0.7, "grid.linestyle": "-",
        "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
        "axes.linewidth": 0.8, "xtick.color": T["axis"], "ytick.color": T["axis"],
        "xtick.labelcolor": T["ink2"], "ytick.labelcolor": T["ink2"], "text.color": T["ink"],
        "legend.frameon": False, "legend.labelcolor": T["ink2"], "legend.fontsize": 9,
        "lines.linewidth": 1.5, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
        "font.family": "sans-serif", "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 9.5, "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.titlepad": 8, "figure.dpi": 110, "figure.constrained_layout.use": True,
        "axes.prop_cycle": mpl.cycler(color=[T["strategy"], T["news"], T["price"], T["ml"]]),
    })


set_theme(False)


# ----------------------------------------------------------------------------- utilitaires
def div_cmap():
    return LinearSegmentedColormap.from_list("div", T["div"])


def seq_cmap():
    return LinearSegmentedColormap.from_list("seq", T["seq"])


def _zero(ax, y=0.0):
    ax.axhline(y, color=T["axis"], lw=0.9, zorder=1)


def _end_label(ax, x, y, text, color):
    ax.scatter([x], [y], s=34, color=color, edgecolor=T["surface"], linewidth=1.5, zorder=6, clip_on=False)
    ax.annotate(text, (x, y), xytext=(7, 0), textcoords="offset points", va="center", fontsize=8.5,
                color=T["ink2"], annotation_clip=False)


def _end_labels(ax, items, min_px=12):
    """items : [(x, y, texte, couleur)] par priorité décroissante. Un point de fin pour chacun ;
    le texte n'est posé que s'il ne chevauche pas une étiquette déjà placée (sinon : la légende suffit)."""
    ax.figure.canvas.draw()  # nécessaire pour convertir en pixels avec l'échelle finale
    placed = []
    for x, y

## 7 · La plateforme : API, tableau de bord, site

- `app/api.py` : une API FastAPI qui sert le signal, l'historique et le prédicteur (on règle les variables, l'API
  renvoie la probabilité et les contributions).
- `app/dashboard.py` : un tableau de bord Streamlit.
- `app/export_site.py` : fabrique le site statique publié sur GitHub Pages : il relit `runs/`, les notebooks et les
  figures, écrit `site/data/donnees.js`, convertit les notebooks en HTML et prépare les archives téléchargeables.
- `app/export_statique.py`, `app/build_site.py` : anciennes versions de l'export.

In [27]:
montrer("app/api.py")

**`app/api.py`** · 254 lignes

"""API du projet Nikkei : le point d'accès qui sert la prédiction (« served prediction endpoint » de l'énoncé).

Lancer depuis la racine du dépôt : uvicorn app.api:app --port 8000
Documentation interactive : http://127.0.0.1:8000/docs
Le site de présentation (dossier web/, en PHP) interroge cette API.
"""
import json
from functools import lru_cache

import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException, Query

from src.config import COLONNES, FEATURES, RUNS
from src.data import charger_table
from src.decision import charger_decision
from src.logreg_scratch import constante_lipschitz, cout, descente, gradient  # annexe de Maths
from src.predict import contributions, historique, signal

app = FastAPI(title="Projet Nikkei", version="1.1",
              description="Signal mensuel acheter / neutre / vendre sur le Nikkei 225 pour un investisseur en euros.")
TABLE = charger_table()


@app.get("/sante")
def sante() -> dict:
    """État du service et de la décision figée."""
    dec = charger_decision()
    return {"statut": "ok", "modele": dec["modele_nikkei"], "decision_calculee_le": dec["date"],
            "dernier_mois_disponible": TABLE.index.max().strftime("%Y-%m")}


@app.get("/mois")
def mois_disponibles() -> list[dict]:
    """Mois pour lesquels un signal peut être demandé (validation, test, mois à prédire), du plus récent au plus ancien."""
    d = TABLE[TABLE["bloc"].isin(["validation", "test", "à prédire"])]
    return [{"mois": date.strftime("%Y-%m"), "bloc": bloc} for date, bloc in zip(d.index[::-1], d["bloc"][::-1])]


@app.get("/signal")
def signal_du_mois(mois: str | None = Query(None, pattern=r"^\d{4}-\d{2}$", description="mois des données, AAAA-MM")) -> dict:
    """Probabilité de hausse du mois suivant, score, action du modèle et recommandation finale."""
    try:
        return signal(mois, TABLE)
    except KeyError as erreur:
        raise HTTPException(status_code=404, detail=str(erreur))


@app.get("/explication")
def explication(mois: str | None = Query(None, pattern=r"^\d{4}-\d{2}$")) -> list[dict]:
    """Contributions SHAP de chaque variable au log-odds de la baisse, de la plus forte à la plus faible."""
    try:
        return contributions(mois, TABLE).round(4).to_dict(orient="records")
    except KeyError as erreur:
        raise HTTPException(status_code=404, detail=str(erreur))


@app.get("/historique")
def historique_des_decisions(bloc: str = Query("test", pattern="^(validation|test)$")) -> list[dict]:
    """Décisions mois par mois et rendements réalisés, sur la validation ou le test."""
    h = historique(TABLE, blocs=(bloc,)).round(6).reset_index()   # 6 décimales : les rendements sont composés côté site
    h["date"] = h["date"].dt.strftime("%Y-%m")
    return h.to_dict(orient="records")


@app.get("/evaluation")
def evaluation() -> dict:
    """Résultats de l'évaluation unique sur le test 2024-2026."""
    return json.loads((RUNS / "evaluation_test.json").read_text())


# Variables d'information que le prédicteur du site laisse régler (les autres gardent leur valeur du mois de base)
REGLABLES = {
    "news_attention_6m": ("Volume anormal d'articles", "ratio"),
    "news_sent_w": ("Sentiment des news (FinBERT)", "score"),
    "n225_mom_12m": ("Momentum du Nikkei sur 12 mois", "pourcent"),
    "n225_mom_3m": ("Momentum du Nikkei sur 3 mois", "pourcent"),
    "d_jgb10y_bp": ("Variation du taux japonais à 10 ans", "pb"),
    "carry_1m": ("Portage de la couverture", "annuel"),
}


def _ligne_de_base(mois: str | None):
    lignes = TABLE[TABLE["bloc"] == "à prédire"] if mois is None else TABLE[TABLE.index.strftime("%Y-%m") == mois]
    if lignes.empty:
        raise HTTPException(status_code=404, detail=f"aucune donnée pour le mois {mois}")
    return lignes.index[0]


@app.get("/variables")
def variables_reglables(mois: str | None = Query(None, pattern=r"^\d{4}-\d{2}$")) -> list[dict]:
    """Variables d'information réglables : libellé, bornes observées sur 

In [28]:
montrer("app/dashboard.py")

**`app/dashboard.py`** · 116 lignes

"""Tableau de bord du projet Nikkei (démo de l'oral).

Lancer depuis la racine du dépôt : streamlit run app/dashboard.py
"""
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.config import FIG, RUNS  # noqa: E402
from src.data import charger_table  # noqa: E402
from src.decision import charger_decision  # noqa: E402
from src.predict import contributions, historique, signal  # noqa: E402

st.set_page_config(page_title="Projet Nikkei", layout="wide")
COULEURS = {"acheter": "#548235", "neutre": "#9e9e9e", "vendre": "#c00000"}


@st.cache_data
def donnees():
    table = charger_table()
    return table, historique(table), json.loads((RUNS / "evaluation_test.json").read_text())


TABLE, HIST, EVAL = donnees()
DEC = charger_decision()
MOIS = [d.strftime("%Y-%m") for d in TABLE.index[TABLE["bloc"].isin(["validation", "test", "à prédire"])]]

st.title("Nikkei 225 : acheter, rester neutre ou vendre ?")
st.caption("Signal mensuel pour un investisseur en euros · régression logistique calibrée · décision figée avant le test")

onglets = st.tabs(["Signal du mois", "Pourquoi ce signal", "Historique", "Le modèle a-t-il marché ?", "Méthode"])

with onglets[0]:
    mois = st.selectbox("Données de fin de mois", MOIS[::-1], index=0)
    s = signal(mois, TABLE)
    st.error(f"**Recommandation finale : {s['recommandation_finale']}.** "
             "Sur le test 2024-2026, le modèle ne bat pas le modèle constant : sa règle est montrée à titre d'analyse.")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Probabilité de hausse le mois suivant", f"{s['p_hausse']:.0%}")
    c2.metric("Score S = 2p − 1", f"{s['score']:+.2f}")
    c3.metric("Action de la règle du modèle", s["action_du_modele"])
    c4.metric("Couverture si achat", "couvert")

    fig, ax = plt.subplots(figsize=(9, 1.3))
    tv, ta = s["seuils"]["theta_v"], s["seuils"]["theta_a"]
    ax.axvspan(0, tv, color=COULEURS["vendre"], alpha=0.25)
    ax.axvspan(tv, ta, color=COULEURS["neutre"], alpha=0.25)
    ax.axvspan(ta, 1, color=COULEURS["acheter"], alpha=0.25)
    ax.axvline(s["p_hausse"], color="black", lw=3)
    for x, texte in [(tv / 2, "vendre"), ((tv + ta) / 2, "neutre"), ((ta + 1) / 2, "acheter")]:
        ax.text(x, 0.5, texte, ha="center", va="center", fontsize=10)
    ax.set_xlim(0, 1); ax.set_yticks([]); ax.set_xlabel("probabilité calibrée de hausse")
    st.pyplot(fig)
    st.caption(f"Seuils tirés de l'espérance de gain en euros : θ_v = {tv:.3f}, θ_a = {ta:.3f} (marge de 0,5 %/mois).")
    if s["rendement_realise_mois_suivant"] is None:
        st.info("Le mois suivant n'est pas encore connu. Attention : la collecte des news de fin 2026 est incomplète.")
    else:
        st.write(f"Ce qui s'est passé ensuite : Nikkei **{100 * s['rendement_realise_mois_suivant']:+.1f} %** le mois suivant.")

with onglets[1]:
    mois_x = st.selectbox("Mois à expliquer", MOIS[::-1], index=0, key="explication")
    c = contributions(mois_x, TABLE).head(8).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(c["variable"], c["contribution"], color=np.where(c["contribution"] > 0, "#c55a11", "#1f4e79"))
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("contribution au log-odds de la baisse (> 0 : pousse vers « baisse »)")
    st.pyplot(fig)
    st.caption("SHAP d'un modèle linéaire : poids × écart de la variable standardisée à sa moyenne d'entraînement. "
               "SHAP décrit le modèle, pas l'économie : il n'établit aucune causalité (S9).")
    st.dataframe(contributions(mois_x, TABLE).round(3), use_container_width=True)

with onglets[2]:
    bloc = st.radio("Période", ["test", "validation"], horizontal=True)
    h = HIST[HIST["bloc"] == bloc]
    cumul = (1 + h[["rendement_regle", "rendement_achat_couvert", "rendement_cash"]]).cumprod()
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(cumul.index, c

In [29]:
montrer("app/export_site.py")

**`app/export_site.py`** · 489 lignes

"""Prépare le site « Visite et Lecture » (dossier site/) : données, figures, notebooks et PDF.

Le site est statique : il ne demande ni PHP ni API vivante. Ce script interroge l'API en mémoire (sans serveur),
écrit toutes les réponses dans site/data/donnees.js, puis copie à côté les figures, les notebooks (en HTML pour les lire,
en .ipynb et en .zip pour les télécharger) et les deux PDF. À relancer après tout changement des résultats ou des notebooks :

    python -m app.export_site

Puis, pour regarder le site : python -m http.server 8095 --directory site  (adresse http://localhost:8095).
"""
import json
import re
import shutil
import subprocess
import warnings
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
SITE = ROOT / "site"

ROUTES = {
    "sante": "/sante",
    "evaluation": "/evaluation",
    "decision": "/decision",
    "serie": "/serie",
    "essais": "/essais",
    "surface": "/surface",
    "volatilite": "/volatilite",
    "signal_risque": "/signal_risque",
    "verifications": "/verifications",
    "signal": "/signal",
    "explication": "/explication",
    "historique_test": "/historique?bloc=test",
    "historique_validation": "/historique?bloc=validation",
}

NOTEBOOKS = [
    ("00a_collecte_V1", "Collecte, première version : prix, taux, change, news"),
    ("00b_collecte_V2", "Collecte, seconde version : 1999-2026, nouvelles sources"),
    ("01_cadrage_eda_variables", "Cadrage, audit des fuites, exploration, variables, découpage"),
    ("02_pipeline_modeles_optuna", "Pipeline, trois modèles, Optuna, journal des essais"),
    ("03_calibration_seuils", "Calibration, choix du modèle, règle de décision figée"),
    ("03b_test_final", "Évaluation unique sur le test 2024-2026"),
    ("04_explicabilite_diagnostics", "Tranches, permutation, SHAP, dépendances partielles"),
    ("05_annexe_maths_logistique", "Annexe de Maths : la régression logistique à la main"),
    ("06_annexe_news_prix", "Annexe : signal news × signal prix, temps de réalisation, backtests, HMM"),
    ("07_seconde_version_bloomberg", "Seconde version : 1999-2026, Bloomberg, résultats et explicabilité"),
    ("08_ameliorations_sans_surapprentissage", "Améliorations sans surapprentissage : protocole figé, jugé sur 2010-2023"),
    ("09_code_et_concepts", "Tout le code du projet, module par module, avec les concepts expliqués"),
]
# Un notebook rangé dans son propre dossier (avec ses scripts et ses figures) se télécharge avec tout son dossier.
DOSSIERS_NB = {"06_annexe_news_prix": "notebooks/06_annexe_news_prix"}
ARCHIVE_NOTEBOOKS = "notebooks_nikkei_225.zip"   # le dossier notebooks/ au complet
ARCHIVE_PROJET = "projet_nikkei_225.zip"         # tout ce qu'il faut pour refaire tourner le projet
# hors de l'archive du projet : le site lui-même, l'ancien site, les données brutes (lourdes, ou sous licence comme Bloomberg)
HORS_PROJET = ("site/", "site_statique/", "web/", "data/1_brutes/", "data/archives/", ".claude/", "lancer.php", "Ouvrir le site.html",
               "Lancer le projet.command")
DATE_FIXE = (2026, 1, 1, 0, 0, 0)               # archives identiques d'un export à l'autre si rien n'a changé
WIDGET = "application/vnd.jupyter.widget-view+json"
NOTE_WIDGET = ('<div style="padding:12px 14px;border:1px solid #d9d2c5;border-radius:4px;background:#f7f3ec;color:#1f1d1a;'
               'font:14px/1.5 system-ui,sans-serif"><b>Tableau de bord interactif (ipywidgets).</b> Ses curseurs ont besoin de Python : '
               'ils ne fonctionnent pas dans cette page. Téléchargez le dossier du notebook depuis le site, puis ouvrez-le dans Jupyter.</div>')

LISEZMOI_PROJET = """# Projet Nikkei 225 : version finale

Cette archive contient tout le projet, sauf le site de présentation et les données brutes :
- `notebooks/` : tous les notebooks, exécutés (le 06 avec ses scripts, ses figures et son README) ;
- `src/`, `app/`, `tests/` : le code, l'API et les tests ;
- `data/2_construites/`, `data/3_modele/` : les tables mensue

In [30]:
montrer("app/export_statique.py")

**`app/export_statique.py`** · 137 lignes

"""Fabrique une copie statique du site, publiable sans serveur (Python et PHP doivent tourner pendant l'export).

Les pages sont rendues par le serveur PHP, les réponses de l'API sont enregistrées en fichiers JSON, et un petit script
injecté en tête de chaque page renvoie les appels « api.php?... » vers ces fichiers. Seules les simulations « et si »
du prédicteur (réglages libres) ont besoin de l'API vivante : dans la copie, elles affichent un message.

Usage : python -m app.export_statique [--site http://localhost:8080] [--sortie site_statique]
"""
import argparse
import json
import re
import shutil
import urllib.parse
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
WEB = ROOT / "web"
NOTEBOOKS = ["01_cadrage_eda_variables", "02_pipeline_modeles_optuna", "03_calibration_seuils", "03b_test_final",
             "04_explicabilite_diagnostics", "05_annexe_maths_logistique"]
SIMPLES = ["sante", "mois", "evaluation", "decision", "serie", "essais", "surface", "volatilite", "verifications", "signal_risque",
           "signal", "explication", "variables", "prediction"]
PAR_MOIS = ["signal", "explication", "variables", "prediction"]

# Redirige les appels à l'API vers les fichiers JSON de la copie
AIGUILLAGE = """<script>
(() => {
  const nomFichier = (u) => {
    const p = new URLSearchParams(u.split("?")[1] || ""), route = p.get("route"); p.delete("route");
    const cles = [...p.keys()].sort();
    if (cles.some((k) => !["mois", "bloc"].includes(k))) return null;              // simulation libre : pas de fichier
    return route + cles.map((k) => "__" + k + "-" + p.get(k)).join("");
  };
  const fetchOriginal = window.fetch.bind(window);
  let paquet = null;                                  // toutes les réponses de l'API, dans un seul fichier chargé une fois
  const donnees = () => (paquet ??= fetchOriginal("donnees.json").then((r) => r.json()));
  const reponse = (corps, statut) => new Response(JSON.stringify(corps), { status: statut, headers: { "Content-Type": "application/json" } });
  window.fetch = (entree, options) => {
    const u = typeof entree === "string" ? entree : entree.url;
    if (u && u.includes("api.php?")) {
      const f = nomFichier(u);
      if (!f) return Promise.resolve(reponse({ detail: "Simulation disponible seulement dans la version locale du site" }, 503));
      return donnees().then((d) => (f in d ? reponse(d[f], 200) : reponse({ detail: "donnée absente de la copie" }, 404)));
    }
    return fetchOriginal(entree, options);
  };
  for (const nom of ["replaceState", "pushState"]) {
    const original = history[nom].bind(history);
    history[nom] = (...a) => { try { original(...a); } catch (e) {} };
  }
})();
</script>
"""


def lire(url: str) -> bytes:
    with urllib.request.urlopen(url, timeout=60) as r:
        return r.read()


def nom_json(route: str, params: dict) -> str:
    return route + "".join(f"__{k}-{params[k]}" for k in sorted(params)) + ".json"


def relier(html: str) -> str:
    """Rend les liens absolus du site relatifs à la copie."""
    html = re.sub(r'href="/notebooks\?nb=([a-z0-9_]+)"', r'href="notebooks-\1.html"', html)
    html = html.replace('href="/notebooks"', 'href="notebooks.html"')
    html = re.sub(r'href="/recit([?#][^"]*)?"', lambda m: f'href="recit.html{m.group(1) or ""}"', html)
    html = html.replace('href="/"', 'href="accueil.html"')
    html = re.sub(r'(href|src)="/(assets|figures|nb)/', r'\1="\2/', html)
    html = re.sub(r'href="/(rapport|rapport_maths)\.pdf"', r'href="\1.pdf"', html)
    html = re.sub(r'<a href="/ipynb/[^"]*"[^>]*>[^<]*</a>', "", html)                # pas de .ipynb dans la copie
    html = re.sub(r'href="http://127\.0\.0\.1:\d+/docs"', 'href="recit.html#ressources"', html)
    return html


def sans_enveloppe(html: str) -> str:
    """Page principale d'un Artifact : sans doctype, html, head ni body (la plateforme les ajoute)."""
    html = re.sub(r"<!doctype[^>]*>", "", html, flags=re.I)
    html = r

In [31]:
montrer("app/build_site.py")

**`app/build_site.py`** · 35 lignes

"""Convertit les notebooks en pages HTML pour le site (web/nb/).

À relancer après toute modification d'un notebook : python -m app.build_site
"""
from pathlib import Path

import nbformat
from nbconvert import HTMLExporter

ROOT = Path(__file__).resolve().parents[1]
SORTIE = ROOT / "web" / "nb"

NOTEBOOKS = [
    ("01_cadrage_eda_variables", "Cadrage, audit des fuites, EDA, variables, split"),
    ("02_pipeline_modeles_optuna", "Pipeline, trois modèles, Optuna, journal des essais"),
    ("03_calibration_seuils", "Calibration, choix du modèle, règle de décision"),
    ("03b_test_final", "Évaluation unique sur le test 2024-2026"),
    ("04_explicabilite_diagnostics", "Slices, permutation, SHAP, PDP"),
    ("05_annexe_maths_logistique", "Annexe de Maths : la logistique à la main"),
]


def main() -> None:
    SORTIE.mkdir(parents=True, exist_ok=True)
    exporteur = HTMLExporter(template_name="lab")
    exporteur.embed_images = True
    for nom, _ in NOTEBOOKS:
        nb = nbformat.read(ROOT / "notebooks" / f"{nom}.ipynb", as_version=4)
        html, _ = exporteur.from_notebook_node(nb, resources={"metadata": {"name": nom}})
        (SORTIE / f"{nom}.html").write_text(html, encoding="utf-8")
        print(f"{nom}.html : {len(html) / 1e6:.1f} Mo")


if __name__ == "__main__":
    main()

## Ce qu'il faut retenir

- Le projet tient en une idée : **ne jamais laisser le futur entrer dans le passé** (décalages de publication, audit
  des fuites, découpage chronologique, pipeline, décision figée, test ouvert une fois, protocole figé pour la V3).
- Le meilleur modèle est une logistique régularisée et calibrée : simple, explicable par SHAP, et pas battue par les
  arbres sur si peu de mois.
- Le sens du Nikkei d'un mois sur l'autre se prévoit à peine ; sa volatilité se prévoit nettement mieux.